# LeetCode 185 — Department Top Three Salaries (Hard, Database) — học SQL từ con số 0

Đây là một bài thuộc mảng **Database** của LeetCode. Khác với các bài thuật toán, ở đây **không có** `class Solution`: ta nộp một **câu truy vấn SQL** (chọn ngôn ngữ *MySQL* trên LeetCode), hoặc — ở tab *Pandas* — một hàm Python có signature:

```python
import pandas as pd

def top_three_salaries(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:

```

Notebook dùng `sqlite3` (có sẵn trong thư viện chuẩn Python) để bạn **chạy SQL thật** ngay trong Jupyter, không cần cài MySQL.

---

## Sau notebook này, bạn sẽ nắm được

1. Các khái niệm gốc của cơ sở dữ liệu: **bảng** (table), **hàng** (row), **cột** (column), **khóa chính** (primary key), **khóa ngoại** (foreign key), giá trị `NULL`.
2. Các câu lệnh SQL nền tảng, mỗi lệnh một ví dụ chạy được: `SELECT`, `WHERE`, `ORDER BY`, `DISTINCT`, `LIMIT`, `JOIN`, `LEFT JOIN`, `COUNT(DISTINCT ...)`, `GROUP BY`, `HAVING`.
3. **Subquery** (truy vấn con) và đặc biệt **correlated subquery** (truy vấn con tương quan) — hiểu nó như một vòng `for` lồng nhau.
4. **Window function** (hàm cửa sổ) và sự khác nhau *rất quan trọng* giữa `ROW_NUMBER`, `RANK`, `DENSE_RANK` khi có lương trùng nhau.
5. Năm cách giải đi từ dễ tới hiện đại, có trace từng bước, test đối chiếu chéo trên dữ liệu ngẫu nhiên.
6. Các lỗi kinh điển (dùng `RANK`, `LIMIT 3`, quên `DISTINCT`, quên phòng ít hơn 3 người...) và 6 bài tập SQL liên quan (#176, #177, #184...).

Lộ trình:

`Python thuần (hiểu đề)` → `Correlated subquery` → `Self-join + GROUP BY + HAVING` → `DENSE_RANK() trong CTE (cách chính)` → `Pandas (tùy chọn)`.

## 1. Đề bài (theo LeetCode)

**Bảng `Employee`**

```text
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| id           | int     |
| name         | varchar |
| salary       | int     |
| departmentId | int     |
+--------------+---------+
```

- `id` là **primary key** (cột có giá trị không trùng nhau) của bảng này.
- `departmentId` là **foreign key** (cột tham chiếu) tới `id` của bảng `Department`.
- Mỗi hàng cho biết `id`, tên, lương của một nhân viên và `id` phòng ban của người đó.

**Bảng `Department`**

```text
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
+-------------+---------+
```

- `id` là primary key của bảng này.
- Mỗi hàng cho biết `id` và tên của một phòng ban.

**Yêu cầu.** Ban lãnh đạo muốn biết ai kiếm nhiều tiền nhất trong mỗi phòng. Một **high earner** của một phòng là nhân viên có mức lương nằm trong **top 3 mức lương phân biệt** (*top three unique salaries*) của phòng đó.

Hãy tìm tất cả high earner của mọi phòng. Trả về bảng kết quả theo **thứ tự bất kỳ**.

> Gốc tiếng Anh: *A company's executives are interested in seeing who earns the most money in each of the company's departments. A high earner in a department is an employee who has a salary in the top three unique salaries for that department. Write a solution to find the employees who are high earners in each of the departments. Return the result table in any order.*

### Example 1

Input — bảng `Employee`:

```text
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 85000  | 1            |
| 2  | Henry | 80000  | 2            |
| 3  | Sam   | 60000  | 2            |
| 4  | Max   | 90000  | 1            |
| 5  | Janet | 69000  | 1            |
| 6  | Randy | 85000  | 1            |
| 7  | Will  | 70000  | 1            |
+----+-------+--------+--------------+
```

Input — bảng `Department`:

```text
+----+-------+
| id | name  |
+----+-------+
| 1  | IT    |
| 2  | Sales |
+----+-------+
```

Output:

```text
+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |
+------------+----------+--------+
```

Giải thích (theo đề):

- Phòng IT: Max có mức lương phân biệt cao nhất; Randy **và** Joe cùng có mức lương phân biệt cao thứ hai; Will có mức lương phân biệt cao thứ ba.
- Phòng Sales: Henry lương cao nhất; Sam lương cao thứ hai; **không có** mức lương cao thứ ba vì phòng chỉ có 2 nhân viên.

### Constraints

- Không có hai nhân viên nào trùng **cả** tên, lương **và** phòng ban.

(Đề không cho giới hạn số hàng. Kết quả phải có đúng 3 cột tên `Department`, `Employee`, `Salary`.)

## 2. Hiểu đề bằng tay trước khi viết bất kỳ dòng SQL nào

Từ khóa quan trọng nhất của đề là **"unique salaries"** — *mức lương phân biệt*. Ta không lấy "3 **người** lương cao nhất", mà lấy "3 **mức lương** cao nhất", rồi lấy **tất cả** những người có một trong ba mức đó.

**Phòng IT** có 5 người: Joe 85000, Max 90000, Janet 69000, Randy 85000, Will 70000.

| Bước | Kết quả |
|---|---|
| Liệt kê lương | 85000, 90000, 69000, 85000, 70000 |
| Bỏ trùng (mức lương phân biệt) | 90000, 85000, 70000, 69000 |
| Lấy 3 mức cao nhất | **90000, 85000, 70000** |
| Ai có lương thuộc 3 mức đó? | Max (90000), Joe (85000), Randy (85000), Will (70000) |

→ IT có **4 người** trong kết quả dù chỉ là "top 3", vì Joe và Randy hòa nhau ở mức 85000. Janet (69000) là mức thứ 4 → bị loại.

**Phòng Sales** có 2 người: Henry 80000, Sam 60000 → chỉ có 2 mức lương, cả 2 mức đều nằm trong "top 3" → lấy cả 2 người.

Hai cái bẫy đã lộ ra ngay từ ví dụ:

1. **Hòa lương** làm số người > 3 (IT có 4 người).
2. **Phòng ít người** làm số người < 3 (Sales có 2 người) — không được bỏ cả phòng chỉ vì "không có hạng 3".

Ta ghi dữ liệu ví dụ vào Python để dùng xuyên suốt notebook. Mỗi nhân viên là một **tuple** `(id, name, salary, departmentId)` — đúng thứ tự cột của bảng.

In [1]:
EMPLOYEES_1 = [
    (1, "Joe",   85000, 1),
    (2, "Henry", 80000, 2),
    (3, "Sam",   60000, 2),
    (4, "Max",   90000, 1),
    (5, "Janet", 69000, 1),
    (6, "Randy", 85000, 1),
    (7, "Will",  70000, 1),
]
DEPARTMENTS_1 = [
    (1, "IT"),
    (2, "Sales"),
]

# Output mong đợi của Example 1 (đề cho phép thứ tự bất kỳ)
EXPECTED_1 = [
    ("IT", "Max", 90000), ("IT", "Joe", 85000), ("IT", "Randy", 85000),
    ("IT", "Will", 70000), ("Sales", "Henry", 80000), ("Sales", "Sam", 60000),
]

print("Số nhân viên:", len(EMPLOYEES_1), "| Số phòng:", len(DEPARTMENTS_1))
print("Hàng đầu tiên của Employee:", EMPLOYEES_1[0])
print("Lương phân biệt của IT:", sorted({s for (_, _, s, d) in EMPLOYEES_1 if d == 1}, reverse=True))

Số nhân viên: 7 | Số phòng: 2
Hàng đầu tiên của Employee: (1, 'Joe', 85000, 1)
Lương phân biệt của IT: [90000, 85000, 70000, 69000]


## 3. Nền tảng: cơ sở dữ liệu quan hệ và SQL

### 3.1. Bảng, hàng, cột, khóa — nói bằng ngôn ngữ đời thường

- **Cơ sở dữ liệu quan hệ** (relational database): nơi lưu dữ liệu dưới dạng nhiều **bảng** có liên hệ với nhau. Hãy tưởng tượng một file Excel có nhiều sheet.
- **Bảng** (table): một sheet, ví dụ `Employee`.
- **Cột** (column): một thuộc tính có tên và kiểu dữ liệu, ví dụ `salary` kiểu `int`. Trong Python, cột ≈ một vị trí trong tuple.
- **Hàng** (row / record): một đối tượng cụ thể, ví dụ nhân viên Joe. Trong Python, hàng ≈ một tuple `(1, "Joe", 85000, 1)`.
- **Khóa chính** (primary key): cột mà giá trị **không bao giờ trùng** giữa các hàng, dùng để chỉ đích danh một hàng. `Employee.id = 4` chắc chắn là Max.
- **Khóa ngoại** (foreign key): cột trỏ sang khóa chính của **bảng khác**. `Employee.departmentId = 1` nghĩa là "phòng có `Department.id = 1`", tức IT.

```text
   Employee                                  Department
+----+-------+--------+--------------+       +----+-------+
| id | name  | salary | departmentId | ----> | id | name  |
+----+-------+--------+--------------+       +----+-------+
| 1  | Joe   | 85000  |      1       | ----> | 1  | IT    |
| 2  | Henry | 80000  |      2       | ----> | 2  | Sales |
+----+-------+--------+--------------+       +----+-------+
```

Vì sao không ghi thẳng tên phòng vào bảng `Employee`? Vì nếu phòng IT đổi tên, ta chỉ cần sửa **một** hàng trong `Department` thay vì sửa hàng trăm hàng nhân viên. Đổi lại, khi muốn in tên phòng cạnh tên nhân viên, ta phải **ghép** (JOIN) hai bảng — đó chính là việc bài này đòi hỏi.

- **SQL** (Structured Query Language): ngôn ngữ để hỏi dữ liệu. Điểm đặc biệt: SQL là ngôn ngữ **khai báo** (declarative) — ta mô tả *muốn kết quả gì*, còn *làm thế nào* (vòng lặp, sắp xếp...) do hệ quản trị cơ sở dữ liệu tự lo.

### 3.2. `sqlite3`: một cơ sở dữ liệu nằm gọn trong Python

`sqlite3` là module có sẵn của Python, chạy SQLite — một cơ sở dữ liệu nhỏ, không cần server. Ba thao tác cần nhớ:

| Lệnh Python | Ý nghĩa |
|---|---|
| `conn = sqlite3.connect(":memory:")` | Mở một cơ sở dữ liệu **trong RAM** (tắt là mất, rất hợp để học) |
| `conn.execute(sql)` | Gửi một câu SQL; trả về một **cursor** (con trỏ đọc kết quả) |
| `cursor.fetchall()` | Lấy toàn bộ kết quả dưới dạng `list` các `tuple` |

**SQLite vs MySQL.** LeetCode chấm bằng MySQL 8. Hai hệ này cùng nói SQL nhưng có "giọng địa phương" (dialect) khác nhau ở vài chỗ; notebook sẽ chỉ rõ mỗi khi gặp. Mọi câu trả lời cuối cùng trong notebook đều được viết sao cho **chạy được trên cả hai**. Window function (mục 8) cần SQLite ≥ 3.25 — ta in phiên bản ra để chắc chắn.

Lệnh đầu tiên: `CREATE TABLE` tạo bảng, liệt kê từng cột kèm kiểu dữ liệu. Ta khai báo `PRIMARY KEY` cho `id` đúng như mô tả đề.

In [2]:
import sqlite3

print("Phiên bản SQLite:", sqlite3.sqlite_version)   # cần >= 3.25 cho window function

conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE Employee (
        id           INT PRIMARY KEY,
        name         VARCHAR(255),
        salary       INT,
        departmentId INT
    )
""")
conn.execute("CREATE TABLE Department (id INT PRIMARY KEY, name VARCHAR(255))")

# sqlite_master là "bảng mục lục" của SQLite: liệt kê các bảng đã tạo
print(conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall())

Phiên bản SQLite: 3.53.1
[('Employee',), ('Department',)]


### 3.3. Thêm dữ liệu: `INSERT INTO ... VALUES`

Câu SQL thêm một hàng:

```sql
INSERT INTO Employee (id, name, salary, departmentId) VALUES (1, 'Joe', 85000, 1);
```

Chuỗi trong SQL đặt trong **dấu nháy đơn** `'Joe'`. Từ Python, ta không tự ghép chuỗi mà dùng **placeholder** `?`: viết `VALUES (?, ?, ?, ?)` rồi truyền tuple giá trị riêng. `executemany` lặp câu đó cho cả danh sách tuple — tiện, và an toàn (không lo dấu nháy trong tên làm hỏng câu lệnh).

Sau đó `SELECT * FROM Employee` nghĩa là "lấy **mọi cột** (`*`) của **mọi hàng** trong `Employee`".

In [3]:
conn.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)", EMPLOYEES_1)
conn.executemany("INSERT INTO Department VALUES (?, ?)", DEPARTMENTS_1)
conn.commit()   # "lưu" thay đổi (với :memory: không bắt buộc, nhưng nên tập thói quen)

cursor = conn.execute("SELECT * FROM Employee")
rows = cursor.fetchall()
print(type(rows), "gồm", len(rows), "hàng")
for row in rows:
    print(row)

<class 'list'> gồm 7 hàng
(1, 'Joe', 85000, 1)
(2, 'Henry', 80000, 2)
(3, 'Sam', 60000, 2)
(4, 'Max', 90000, 1)
(5, 'Janet', 69000, 1)
(6, 'Randy', 85000, 1)
(7, 'Will', 70000, 1)


### 3.4. Công cụ in bảng giống LeetCode

`fetchall()` trả về tuple trần, khó đọc. `cursor.description` cho biết **tên cột** của kết quả (phần tử thứ 0 của mỗi mục). Ta viết vài hàm tiện ích dùng cho cả notebook:

- `make_db(employees, departments)`: tạo một cơ sở dữ liệu mới, đúng schema đề, đổ dữ liệu vào.
- `run(conn, sql)`: chạy SQL, trả về `(tên_cột, danh_sách_hàng)`.
- `show(conn, sql)`: in kết quả thành bảng có viền giống LeetCode; `None` của Python (tức `NULL` của SQL) được in là `null`.
- `check_query(...)`: so kết quả với đáp án mong đợi. Vì đề cho phép **thứ tự bất kỳ**, ta **sắp xếp cả hai phía** trước khi so — đây chính là cách hệ thống chấm "any order" làm.

In [4]:
def make_db(employees, departments):
    """Tạo database trong RAM đúng schema LeetCode 185 và đổ dữ liệu."""
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE Employee (id INT PRIMARY KEY, name VARCHAR(255), "
               "salary INT, departmentId INT)")
    db.execute("CREATE TABLE Department (id INT PRIMARY KEY, name VARCHAR(255))")
    db.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)", employees)
    db.executemany("INSERT INTO Department VALUES (?, ?)", departments)
    db.commit()
    return db


def run(db, sql, params=()):
    cur = db.execute(sql, params)
    cols = [d[0] for d in cur.description]
    return cols, cur.fetchall()


def show(db, sql, params=()):
    cols, rows = run(db, sql, params)
    text = [[("null" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in text]) for i, c in enumerate(cols)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = lambda vals: "| " + " | ".join(v.ljust(w) for v, w in zip(vals, widths)) + " |"
    print(line); print(fmt(cols)); print(line)
    for r in text:
        print(fmt(r))
    print(line)


RESULT_COLS = ["Department", "Employee", "Salary"]

def check_query(name, db, sql, expected, cols=RESULT_COLS):
    got_cols, got_rows = run(db, sql)
    assert got_cols == cols, (name, "sai tên cột", got_cols)
    assert sorted(got_rows) == sorted(expected), (name, sorted(got_rows), sorted(expected))
    print(f"{name}: PASS ({len(got_rows)} hàng)")


db1 = make_db(EMPLOYEES_1, DEPARTMENTS_1)
show(db1, "SELECT * FROM Employee")
show(db1, "SELECT * FROM Department")

+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 85000  | 1            |
| 2  | Henry | 80000  | 2            |
| 3  | Sam   | 60000  | 2            |
| 4  | Max   | 90000  | 1            |
| 5  | Janet | 69000  | 1            |
| 6  | Randy | 85000  | 1            |
| 7  | Will  | 70000  | 1            |
+----+-------+--------+--------------+
+----+-------+
| id | name  |
+----+-------+
| 1  | IT    |
| 2  | Sales |
+----+-------+


### 3.5. `SELECT` chọn cột và `AS` đặt tên mới (alias)

```sql
SELECT name AS Employee, salary AS Salary
FROM Employee;
```

- `SELECT` liệt kê **những cột muốn lấy**, cách nhau bởi dấu phẩy.
- `AS` đổi tên cột trong **kết quả** (không đổi bảng gốc). Bài 185 **bắt buộc** cột kết quả tên `Department`, `Employee`, `Salary` → ta sẽ luôn cần `AS`.
- `FROM` cho biết đọc từ bảng nào.

Liên hệ Python: giống `[(name, salary) for (id, name, salary, dep) in EMPLOYEES_1]`.

### 3.6. `WHERE` lọc hàng

`WHERE <điều kiện>` giữ lại **chỉ những hàng** làm điều kiện đúng. Có thể nối điều kiện bằng `AND`, `OR`; so sánh bằng `=` (SQL dùng **một** dấu bằng, không phải `==`), `<>` hoặc `!=`, `<`, `>`, `<=`, `>=`.

Ví dụ: nhân viên phòng 1 (IT) có lương từ 80000 trở lên. Bằng tay: Joe 85000, Max 90000, Randy 85000 → 3 hàng.

In [5]:
show(db1, "SELECT name AS Employee, salary AS Salary FROM Employee")

show(db1, """
    SELECT id, name, salary
    FROM Employee
    WHERE departmentId = 1 AND salary >= 80000
""")

+----------+--------+
| Employee | Salary |
+----------+--------+
| Joe      | 85000  |
| Henry    | 80000  |
| Sam      | 60000  |
| Max      | 90000  |
| Janet    | 69000  |
| Randy    | 85000  |
| Will     | 70000  |
+----------+--------+
+----+-------+--------+
| id | name  | salary |
+----+-------+--------+
| 1  | Joe   | 85000  |
| 4  | Max   | 90000  |
| 6  | Randy | 85000  |
+----+-------+--------+


### 3.7. `ORDER BY` sắp xếp và `DISTINCT` bỏ trùng

- `ORDER BY cột` sắp **tăng dần** (mặc định, có thể ghi rõ `ASC`); thêm `DESC` để sắp **giảm dần**.
- Có thể sắp theo nhiều cột: `ORDER BY departmentId, salary DESC` = "theo phòng trước, trong cùng phòng thì lương cao lên đầu".
- `SELECT DISTINCT ...` bỏ các hàng **trùng hoàn toàn** trong kết quả. `SELECT DISTINCT departmentId, salary` chính là "các mức lương phân biệt của từng phòng" — khái niệm cốt lõi của bài.

Dự đoán trước: IT có 5 người nhưng chỉ có 4 mức lương phân biệt (85000 xuất hiện 2 lần) → truy vấn thứ hai in 4 + 2 = 6 hàng.

In [6]:
show(db1, "SELECT name, departmentId, salary FROM Employee ORDER BY departmentId, salary DESC")
show(db1, "SELECT DISTINCT departmentId, salary FROM Employee ORDER BY departmentId, salary DESC")

+-------+--------------+--------+
| name  | departmentId | salary |
+-------+--------------+--------+
| Max   | 1            | 90000  |
| Joe   | 1            | 85000  |
| Randy | 1            | 85000  |
| Will  | 1            | 70000  |
| Janet | 1            | 69000  |
| Henry | 2            | 80000  |
| Sam   | 2            | 60000  |
+-------+--------------+--------+
+--------------+--------+
| departmentId | salary |
+--------------+--------+
| 1            | 90000  |
| 1            | 85000  |
| 1            | 70000  |
| 1            | 69000  |
| 2            | 80000  |
| 2            | 60000  |
+--------------+--------+


### 3.8. `LIMIT` / `OFFSET` — và cái bẫy đầu tiên của bài

- `LIMIT k` chỉ giữ `k` hàng **đầu tiên** của kết quả (thường dùng sau `ORDER BY`).
- `LIMIT k OFFSET m` bỏ qua `m` hàng đầu rồi lấy `k` hàng.

Người mới rất hay nghĩ: "top 3 thì `ORDER BY salary DESC LIMIT 3`". Nhưng `LIMIT` cắt **toàn bộ kết quả**, không hề biết khái niệm "mỗi phòng", cũng không biết "mức lương phân biệt". Xem điều gì xảy ra:

In [7]:
show(db1, "SELECT name, departmentId, salary FROM Employee ORDER BY salary DESC LIMIT 3")
print("-> Chỉ 3 hàng cho CẢ công ty: thiếu Will (IT) và toàn bộ phòng Sales.")

# LIMIT kết hợp DISTINCT + OFFSET lại rất hữu ích để lấy "mức lương cao thứ k" của MỘT phòng:
show(db1, "SELECT DISTINCT salary FROM Employee WHERE departmentId = 1 "
          "ORDER BY salary DESC LIMIT 1 OFFSET 2")
print("-> 70000 là mức lương phân biệt cao thứ 3 của IT (bỏ qua 2 mức đầu).")

+-------+--------------+--------+
| name  | departmentId | salary |
+-------+--------------+--------+
| Max   | 1            | 90000  |
| Joe   | 1            | 85000  |
| Randy | 1            | 85000  |
+-------+--------------+--------+
-> Chỉ 3 hàng cho CẢ công ty: thiếu Will (IT) và toàn bộ phòng Sales.
+--------+
| salary |
+--------+
| 70000  |
+--------+
-> 70000 là mức lương phân biệt cao thứ 3 của IT (bỏ qua 2 mức đầu).


### 3.9. `JOIN`: ghép hai bảng theo khóa

Kết quả cần **tên phòng** (`IT`, `Sales`) nhưng bảng `Employee` chỉ có `departmentId`. `JOIN` ghép mỗi hàng `Employee` với hàng `Department` thỏa điều kiện `ON`:

```sql
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e
JOIN Department d ON e.departmentId = d.id;
```

- `Employee e` đặt **bí danh bảng** (table alias) `e` cho gọn; từ đó viết `e.name`, `e.salary`.
- Cả hai bảng đều có cột `name`, nên **bắt buộc** ghi rõ `e.name` hay `d.name`, nếu không SQL báo lỗi "ambiguous column".
- `JOIN` (đầy đủ là `INNER JOIN`) chỉ giữ những cặp **khớp** điều kiện. Cách hình dung bằng Python: hai vòng `for` lồng nhau, giữ cặp `(e, d)` nếu `e.departmentId == d.id`.

In [8]:
show(db1, """
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
    ORDER BY Department, Salary DESC
""")

# Cùng ý nghĩa bằng Python (vòng for lồng nhau):
joined = [(dn, en, s) for (_, en, s, dep) in EMPLOYEES_1 for (did, dn) in DEPARTMENTS_1 if dep == did]
print(len(joined), "cặp khớp, ví dụ:", joined[:2])

+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| IT         | Janet    | 69000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |
+------------+----------+--------+
7 cặp khớp, ví dụ: [('IT', 'Joe', 85000), ('Sales', 'Henry', 80000)]


### 3.10. Hàm gộp (aggregate), `GROUP BY` và `HAVING`

**Hàm gộp** biến nhiều hàng thành **một giá trị**:

| Hàm | Ý nghĩa | Với lương IT `85000, 90000, 69000, 85000, 70000` |
|---|---|---|
| `COUNT(*)` | đếm số hàng | 5 |
| `COUNT(salary)` | đếm số giá trị **khác NULL** | 5 |
| `COUNT(DISTINCT salary)` | đếm số giá trị **phân biệt** (khác NULL) | 4 |
| `MAX(salary)` | giá trị lớn nhất | 90000 |

`GROUP BY departmentId` chia các hàng thành **nhóm** theo phòng, rồi hàm gộp tính **riêng cho từng nhóm** → mỗi nhóm thành đúng một hàng kết quả. Tương đương Python: tạo `dict` phòng → danh sách lương, rồi tính trên từng danh sách.

`HAVING` là "`WHERE` dành cho nhóm": lọc **sau khi** đã gộp, nên được phép dùng hàm gộp. Còn `WHERE` lọc **từng hàng trước khi** gộp, nên **không** được viết `WHERE COUNT(*) > 2`.

Dự đoán: IT → 5 người, 4 mức, max 90000; Sales → 2 người, 2 mức, max 80000. Câu thứ hai chỉ giữ phòng có ít nhất 3 mức lương phân biệt → chỉ còn IT.

In [9]:
show(db1, """
    SELECT departmentId,
           COUNT(*)               AS so_nguoi,
           COUNT(DISTINCT salary) AS so_muc_luong,
           MAX(salary)            AS luong_max
    FROM Employee
    GROUP BY departmentId
""")

show(db1, """
    SELECT departmentId, COUNT(DISTINCT salary) AS so_muc_luong
    FROM Employee
    GROUP BY departmentId
    HAVING COUNT(DISTINCT salary) >= 3
""")

+--------------+----------+--------------+-----------+
| departmentId | so_nguoi | so_muc_luong | luong_max |
+--------------+----------+--------------+-----------+
| 1            | 5        | 4            | 90000     |
| 2            | 2        | 2            | 80000     |
+--------------+----------+--------------+-----------+
+--------------+--------------+
| departmentId | so_muc_luong |
+--------------+--------------+
| 1            | 4            |
+--------------+--------------+


### 3.11. `NULL` và `LEFT JOIN`

`NULL` nghĩa là "**không có giá trị / không biết**". Nó có vài tính chất lạ mà ta sẽ cần ở cách 3 và ở phần lỗi thường gặp:

- Mọi phép so sánh với `NULL` cho ra `NULL` (không phải `TRUE`, cũng không phải `FALSE`): `NULL > 5` → `NULL`, thậm chí `NULL = NULL` → `NULL`. Muốn kiểm tra phải dùng `IS NULL`.
- `WHERE` chỉ giữ hàng có điều kiện **đúng hẳn** (`TRUE`) → hàng có điều kiện `NULL` bị **loại âm thầm**.
- `COUNT(cột)` và `COUNT(DISTINCT cột)` **bỏ qua** `NULL`. Còn `COUNT(*)` đếm cả hàng.

**`LEFT JOIN`** giữ **mọi** hàng của bảng bên trái; hàng nào không tìm được bạn ghép thì các cột bên phải được điền `NULL`. Ví dụ dưới: thêm phòng `HR` không có nhân viên. `JOIN` thường bỏ mất HR, `LEFT JOIN` (Department bên trái) vẫn giữ HR với `NULL`.

In [10]:
show(db1, "SELECT NULL > 5 AS a, NULL = NULL AS b, NULL IS NULL AS c, COUNT(NULL) AS d")

db_null = make_db(EMPLOYEES_1[:3], DEPARTMENTS_1 + [(3, "HR")])   # Joe, Henry, Sam + phòng HR trống
print("INNER JOIN:")
show(db_null, "SELECT d.name, e.name FROM Department d JOIN Employee e ON e.departmentId = d.id")
print("LEFT JOIN:")
show(db_null, """SELECT d.name, e.name, COUNT(e.id) OVER (PARTITION BY d.id) AS so_nv
                 FROM Department d LEFT JOIN Employee e ON e.departmentId = d.id""")

+------+------+---+---+
| a    | b    | c | d |
+------+------+---+---+
| null | null | 1 | 0 |
+------+------+---+---+
INNER JOIN:
+-------+-------+
| name  | name  |
+-------+-------+
| IT    | Joe   |
| Sales | Henry |
| Sales | Sam   |
+-------+-------+
LEFT JOIN:
+-------+-------+-------+
| name  | name  | so_nv |
+-------+-------+-------+
| IT    | Joe   | 1     |
| Sales | Henry | 2     |
| Sales | Sam   | 2     |
| HR    | null  | 0     |
+-------+-------+-------+


(Ở cột `so_nv` mình lỡ "nhá hàng" một window function — mục 8 sẽ giảng kỹ. Chỉ cần thấy: HR có `COUNT(e.id) = 0` vì `e.id` là `NULL`.)

### 3.12. Thứ tự **thực thi logic** của một câu SQL

Ta *viết* `SELECT` đầu tiên, nhưng cơ sở dữ liệu *xử lý* theo thứ tự khác. Nắm bảng này giúp tránh rất nhiều lỗi:

| Bước | Mệnh đề | Làm gì |
|:---:|---|---|
| 1 | `FROM` / `JOIN ... ON` | Xác định và ghép các bảng |
| 2 | `WHERE` | Lọc từng hàng |
| 3 | `GROUP BY` | Chia nhóm |
| 4 | `HAVING` | Lọc nhóm |
| 5 | `SELECT` | Tính các cột kết quả — **window function được tính ở đây** |
| 6 | `DISTINCT` | Bỏ hàng trùng |
| 7 | `ORDER BY` | Sắp xếp |
| 8 | `LIMIT` / `OFFSET` | Cắt bớt |

Hệ quả quan trọng cho bài này:

- `LIMIT` chạy **cuối cùng** trên toàn bộ kết quả → không thể dùng nó để lấy "3 mức mỗi phòng".
- Window function tính ở bước 5, **sau** `WHERE` (bước 2) → không thể viết `WHERE DENSE_RANK() OVER (...) <= 3`. Ta sẽ phải bọc nó trong một truy vấn con / CTE.

## 4. Cách 1 — Suy nghĩ bằng Python thuần (để hiểu đề thật chắc)

Trước khi viết SQL, ta giải bài bằng thứ quen thuộc nhất: Python. Cách người mới tự nghĩ ra thường là đúng ba bước làm tay ở mục 2:

1. **Gom lương theo phòng**: `dict` phòng → `set` các mức lương (`set` tự bỏ trùng).
2. **Lấy top 3 mức của mỗi phòng**: `sorted(..., reverse=True)[:3]`.
3. **Duyệt từng nhân viên**: giữ người nào có lương nằm trong top 3 của phòng mình, ghép tên phòng vào.

Ba công cụ Python cần dùng — thử nhanh trên lương phòng IT:

- `set(...)`: tập hợp, tự loại phần tử trùng; kiểm tra `x in s` nhanh (trung bình `O(1)`).
- `sorted(s, reverse=True)`: trả về **list mới** đã sắp giảm dần.
- `lst[:3]`: slicing lấy tối đa 3 phần tử đầu — nếu list chỉ có 2 phần tử thì lấy 2, **không báo lỗi**. Chính tính chất này xử lý tự nhiên phòng ít người như Sales.
- `d.setdefault(k, set())`: nếu `k` chưa có trong dict thì tạo giá trị `set()` rỗng, rồi trả về giá trị đó.

In [11]:
it_salaries = [85000, 90000, 69000, 85000, 70000]
s = set(it_salaries)
print("set:", s)
print("sorted giảm dần:", sorted(s, reverse=True))
print("top 3:", sorted(s, reverse=True)[:3])
print("list 2 phần tử [:3] ->", [80000, 60000][:3], "(không lỗi)")

by_dept = {}
for dep, sal in [(1, 85000), (2, 80000), (1, 90000), (1, 85000)]:
    by_dept.setdefault(dep, set()).add(sal)
print("dict phòng -> set lương:", by_dept)

set: {85000, 70000, 69000, 90000}
sorted giảm dần: [90000, 85000, 70000, 69000]
top 3: [90000, 85000, 70000]
list 2 phần tử [:3] -> [80000, 60000] (không lỗi)
dict phòng -> set lương: {1: {85000, 90000}, 2: {80000}}


In [12]:
def top_three_python(employees, departments):
    """employees: list (id, name, salary, departmentId); departments: list (id, name).
    Trả về list (Department, Employee, Salary)."""
    dept_name = {did: dname for did, dname in departments}

    salaries_by_dept = {}                                   # Bước 1
    for _, _, salary, dep in employees:
        salaries_by_dept.setdefault(dep, set()).add(salary)

    top3 = {dep: set(sorted(sals, reverse=True)[:3])        # Bước 2
            for dep, sals in salaries_by_dept.items()}

    result = []                                             # Bước 3
    for _, name, salary, dep in employees:
        if dep in dept_name and salary in top3[dep]:
            result.append((dept_name[dep], name, salary))
    return result


res = top_three_python(EMPLOYEES_1, DEPARTMENTS_1)
for r in res:
    print(r)
assert sorted(res) == sorted(EXPECTED_1)
print("Khớp Example 1: PASS")

('IT', 'Joe', 85000)
('Sales', 'Henry', 80000)
('Sales', 'Sam', 60000)
('IT', 'Max', 90000)
('IT', 'Randy', 85000)
('IT', 'Will', 70000)
Khớp Example 1: PASS


### Trace cách 1 trên Example 1

**Bước 1–2** (theo từng phòng):

| Phòng | Lương gom được | `set` (bỏ trùng) | Sắp giảm dần | Top 3 |
|---|---|---|---|---|
| 1 (IT) | 85000, 90000, 69000, 85000, 70000 | 4 mức | 90000, 85000, 70000, 69000 | {90000, 85000, 70000} |
| 2 (Sales) | 80000, 60000 | 2 mức | 80000, 60000 | {80000, 60000} |

**Bước 3** (theo từng nhân viên, thứ tự `id`):

| id | name | salary | Phòng | Có trong top 3? | Quyết định |
|:---:|---|---:|---|:---:|---|
| 1 | Joe | 85000 | IT | ✔ | giữ |
| 2 | Henry | 80000 | Sales | ✔ | giữ |
| 3 | Sam | 60000 | Sales | ✔ | giữ |
| 4 | Max | 90000 | IT | ✔ | giữ |
| 5 | Janet | 69000 | IT | ✘ | **loại** (mức thứ 4) |
| 6 | Randy | 85000 | IT | ✔ | giữ |
| 7 | Will | 70000 | IT | ✔ | giữ |

Hàm `trace_python` dưới đây in đúng các trạng thái đó.

In [13]:
def trace_python(employees, departments):
    dept_name = dict(departments)
    salaries_by_dept = {}
    for _, _, salary, dep in employees:
        salaries_by_dept.setdefault(dep, set()).add(salary)
    top3 = {}
    for dep, sals in salaries_by_dept.items():
        ordered = sorted(sals, reverse=True)
        top3[dep] = set(ordered[:3])
        print(f"Phòng {dept_name[dep]:5}: sắp giảm = {ordered} -> top3 = {ordered[:3]}")
    print()
    for eid, name, salary, dep in employees:
        keep = salary in top3[dep]
        print(f"id={eid} {name:5} {salary} ({dept_name[dep]:5}) -> {'giữ' if keep else 'LOẠI'}")


trace_python(EMPLOYEES_1, DEPARTMENTS_1)

Phòng IT   : sắp giảm = [90000, 85000, 70000, 69000] -> top3 = [90000, 85000, 70000]
Phòng Sales: sắp giảm = [80000, 60000] -> top3 = [80000, 60000]

id=1 Joe   85000 (IT   ) -> giữ
id=2 Henry 80000 (Sales) -> giữ
id=3 Sam   60000 (Sales) -> giữ
id=4 Max   90000 (IT   ) -> giữ
id=5 Janet 69000 (IT   ) -> LOẠI
id=6 Randy 85000 (IT   ) -> giữ
id=7 Will  70000 (IT   ) -> giữ


### Vì sao không dừng ở Python?

Cách 1 đúng và dễ hiểu, nhưng:

- LeetCode (tab MySQL) yêu cầu **câu SQL**. Ngoài đời, dữ liệu nằm trong database hàng triệu hàng; kéo hết về Python để lọc là lãng phí — ta muốn database tự lọc và chỉ gửi về vài hàng kết quả.
- SQL không có "biến `dict`" hay "vòng `for`" theo kiểu Python. Ta cần **diễn đạt lại** điều kiện "lương nằm trong top 3 mức phân biệt" thành một thứ SQL hiểu được: một **phép đếm** hoặc một **thứ hạng**.

Độ phức tạp cách 1 (với `n` = số nhân viên): gom `O(n)`, sắp xếp các tập lương tổng cộng `O(n log n)` trường hợp xấu nhất, duyệt lọc `O(n)` → **Time `O(n log n)`**, **Space `O(n)`**.

## 5. Bước đột phá: đổi "top 3" thành "đếm số mức lương cao hơn"

Nhìn lại phòng IT với 4 mức lương phân biệt `90000 > 85000 > 70000 > 69000`. Với mỗi nhân viên, hỏi: **"trong phòng mình, có bao nhiêu mức lương phân biệt *cao hơn* lương mình?"**

| Nhân viên | Lương | Các mức phân biệt cao hơn | Số mức cao hơn | Hạng mức lương |
|---|---:|---|:---:|:---:|
| Max | 90000 | (không có) | 0 | 1 |
| Joe | 85000 | 90000 | 1 | 2 |
| Randy | 85000 | 90000 | 1 | 2 |
| Will | 70000 | 90000, 85000 | 2 | 3 |
| Janet | 69000 | 90000, 85000, 70000 | 3 | 4 |

Quy luật: **hạng mức lương = 1 + số mức phân biệt cao hơn**. Do đó:

> **Lương của `e` nằm trong top 3 mức phân biệt ⇔ số mức lương phân biệt cao hơn `e` trong cùng phòng `< 3`.**

Điều kiện mới này cực kỳ "SQL-friendly": nó chỉ cần **`COUNT(DISTINCT salary)`** trên những người **cùng phòng** và **lương lớn hơn** — toàn những thứ ta vừa học. Hai chi tiết cần nhớ:

- Đếm **`DISTINCT`**: Will thấy 3 *người* cao hơn (Max, Joe, Randy) nhưng chỉ 2 *mức*. Đếm người thì Will bị loại oan.
- Dùng **`< 3`**, không phải `<= 3`: người ở mức thứ 3 có đúng 2 mức cao hơn.

Kiểm chứng quy luật bằng Python trước khi tin nó:

In [14]:
def higher_distinct_count(employees, emp):
    """Số mức lương phân biệt LỚN HƠN lương của emp, trong cùng phòng với emp."""
    _, _, salary, dep = emp
    return len({s for (_, _, s, d) in employees if d == dep and s > salary})


for emp in EMPLOYEES_1:
    k = higher_distinct_count(EMPLOYEES_1, emp)
    print(f"{emp[1]:5} lương {emp[2]} -> {k} mức cao hơn -> hạng {k + 1} -> {'giữ' if k < 3 else 'LOẠI'}")

by_count = [e for e in EMPLOYEES_1 if higher_distinct_count(EMPLOYEES_1, e) < 3]
assert sorted(e[1] for e in by_count) == sorted(r[1] for r in EXPECTED_1)
print("Điều kiện 'đếm < 3' cho đúng tập nhân viên: PASS")

Joe   lương 85000 -> 1 mức cao hơn -> hạng 2 -> giữ
Henry lương 80000 -> 0 mức cao hơn -> hạng 1 -> giữ
Sam   lương 60000 -> 1 mức cao hơn -> hạng 2 -> giữ
Max   lương 90000 -> 0 mức cao hơn -> hạng 1 -> giữ
Janet lương 69000 -> 3 mức cao hơn -> hạng 4 -> LOẠI
Randy lương 85000 -> 1 mức cao hơn -> hạng 2 -> giữ
Will  lương 70000 -> 2 mức cao hơn -> hạng 3 -> giữ
Điều kiện 'đếm < 3' cho đúng tập nhân viên: PASS


## 6. Cách 2 — Correlated subquery (truy vấn con tương quan)

### 6.1. Subquery là gì?

**Subquery** (truy vấn con) là một câu `SELECT` nằm **bên trong** câu SQL khác, đặt trong ngoặc tròn. Nếu nó trả về đúng **một giá trị** (một hàng, một cột) thì gọi là **scalar subquery** và có thể dùng như một con số bình thường.

Ví dụ: "ai có lương cao nhất công ty?" — truy vấn con `(SELECT MAX(salary) FROM Employee)` cho ra `90000`, rồi truy vấn ngoài so sánh với số đó. Truy vấn con này **độc lập**: nó chạy một lần, không phụ thuộc hàng nào bên ngoài.

In [15]:
show(db1, "SELECT MAX(salary) AS max_toan_cong_ty FROM Employee")

show(db1, """
    SELECT name, salary
    FROM Employee
    WHERE salary = (SELECT MAX(salary) FROM Employee)
""")

+------------------+
| max_toan_cong_ty |
+------------------+
| 90000            |
+------------------+
+------+--------+
| name | salary |
+------+--------+
| Max  | 90000  |
+------+--------+


### 6.2. Correlated subquery: truy vấn con "nhìn ra" hàng bên ngoài

Một subquery là **correlated** (tương quan) khi nó dùng cột của **truy vấn bên ngoài**. Khi đó, về mặt ý nghĩa, nó được tính lại **cho từng hàng** của truy vấn ngoài — giống hệt vòng `for` lồng nhau:

```text
for e1 in Employee:                       -- truy vấn ngoài
    for e2 in Employee:                   -- truy vấn con, nhìn thấy e1
        ... dùng e1.departmentId ...
```

Vì cùng đọc bảng `Employee` hai lần, ta **bắt buộc** đặt hai bí danh khác nhau (`e1` cho ngoài, `e2` cho trong) để phân biệt "mình" và "người khác".

Ví dụ khởi động (chưa phải lời giải): ai có lương **cao hơn trung bình phòng mình**? Truy vấn con tính `AVG(salary)` của **chính phòng của `e1`**. Bằng tay: trung bình IT = (85000+90000+69000+85000+70000)/5 = 79800 → Joe, Max, Randy; trung bình Sales = 70000 → Henry.

In [16]:
show(db1, """
    SELECT e1.name, e1.departmentId, e1.salary
    FROM Employee e1
    WHERE e1.salary > (SELECT AVG(e2.salary)
                       FROM Employee e2
                       WHERE e2.departmentId = e1.departmentId)   -- <- tham chiếu ra ngoài
    ORDER BY e1.departmentId, e1.salary DESC
""")

+-------+--------------+--------+
| name  | departmentId | salary |
+-------+--------------+--------+
| Max   | 1            | 90000  |
| Joe   | 1            | 85000  |
| Randy | 1            | 85000  |
| Henry | 2            | 80000  |
+-------+--------------+--------+


### 6.3. Áp dụng vào bài: "trace" bằng chính SQL

Ta đặt điều kiện ở mục 5 thành một truy vấn con tương quan:

```sql
(SELECT COUNT(DISTINCT e2.salary)
 FROM Employee e2
 WHERE e2.departmentId = e1.departmentId     -- cùng phòng với e1
   AND e2.salary > e1.salary)                -- lương cao hơn e1
```

Mẹo học SQL rất hữu ích: trước khi **lọc** bằng một biểu thức, hãy **in** biểu thức đó ra thành một cột để "trace". Dưới đây cột `higher_cnt` là số mức cao hơn của từng người — so với bảng ở mục 5 xem có khớp không.

In [17]:
show(db1, """
    SELECT e1.name, e1.departmentId AS dep, e1.salary,
           (SELECT COUNT(DISTINCT e2.salary)
            FROM Employee e2
            WHERE e2.departmentId = e1.departmentId
              AND e2.salary > e1.salary) AS higher_cnt
    FROM Employee e1
    ORDER BY dep, e1.salary DESC
""")

+-------+-----+--------+------------+
| name  | dep | salary | higher_cnt |
+-------+-----+--------+------------+
| Max   | 1   | 90000  | 0          |
| Joe   | 1   | 85000  | 1          |
| Randy | 1   | 85000  | 1          |
| Will  | 1   | 70000  | 2          |
| Janet | 1   | 69000  | 3          |
| Henry | 2   | 80000  | 0          |
| Sam   | 2   | 60000  | 1          |
+-------+-----+--------+------------+


Khớp hoàn toàn: Max 0, Joe 1, Randy 1, Will 2, **Janet 3**, Henry 0, Sam 1. Giờ chỉ cần chuyển biểu thức đó vào `WHERE ... < 3`, rồi `JOIN Department` để lấy tên phòng và đặt alias cột đúng đề:

In [18]:
SQL_CORRELATED = """
SELECT d.name AS Department, e1.name AS Employee, e1.salary AS Salary
FROM Employee e1
JOIN Department d ON e1.departmentId = d.id
WHERE (SELECT COUNT(DISTINCT e2.salary)
       FROM Employee e2
       WHERE e2.departmentId = e1.departmentId
         AND e2.salary > e1.salary) < 3
"""

show(db1, SQL_CORRELATED)
check_query("Cách 2 - correlated subquery", db1, SQL_CORRELATED, EXPECTED_1)

+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Joe      | 85000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |
| IT         | Max      | 90000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
+------------+----------+--------+
Cách 2 - correlated subquery: PASS (6 hàng)


### 6.4. Trace "vòng for lồng nhau" của correlated subquery

Để thấy rõ truy vấn con được tính lại **cho từng `e1`**, hàm `trace_correlated` mô phỏng đúng logic đó bằng Python: với mỗi `e1`, quét tất cả `e2`, giữ những `e2` cùng phòng và lương cao hơn, rồi đếm số **mức** phân biệt.

| e1 | Những `e2` thỏa điều kiện (cùng phòng, lương cao hơn) | `COUNT(DISTINCT e2.salary)` | `< 3`? |
|---|---|:---:|:---:|
| Joe 85000 (IT) | Max 90000 | 1 | ✔ |
| Henry 80000 (Sales) | — | 0 | ✔ |
| Sam 60000 (Sales) | Henry 80000 | 1 | ✔ |
| Max 90000 (IT) | — | 0 | ✔ |
| Janet 69000 (IT) | Joe 85000, Max 90000, Randy 85000, Will 70000 | 3 | ✘ |
| Randy 85000 (IT) | Max 90000 | 1 | ✔ |
| Will 70000 (IT) | Joe 85000, Max 90000, Randy 85000 | **2** (3 người nhưng 2 mức) | ✔ |

In [19]:
def trace_correlated(employees):
    for e1 in employees:                           # truy vấn ngoài
        matched = [e2 for e2 in employees          # truy vấn con, nhìn thấy e1
                   if e2[3] == e1[3] and e2[2] > e1[2]]
        cnt = len({e2[2] for e2 in matched})       # COUNT(DISTINCT e2.salary)
        who = ", ".join(f"{e2[1]} {e2[2]}" for e2 in matched) or "-"
        print(f"e1={e1[1]:5} {e1[2]} | e2 thỏa: {who:44} | cnt={cnt} -> {'giữ' if cnt < 3 else 'LOẠI'}")


trace_correlated(EMPLOYEES_1)

e1=Joe   85000 | e2 thỏa: Max 90000                                    | cnt=1 -> giữ
e1=Henry 80000 | e2 thỏa: -                                            | cnt=0 -> giữ
e1=Sam   60000 | e2 thỏa: Henry 80000                                  | cnt=1 -> giữ
e1=Max   90000 | e2 thỏa: -                                            | cnt=0 -> giữ
e1=Janet 69000 | e2 thỏa: Joe 85000, Max 90000, Randy 85000, Will 70000 | cnt=3 -> LOẠI
e1=Randy 85000 | e2 thỏa: Max 90000                                    | cnt=1 -> giữ
e1=Will  70000 | e2 thỏa: Joe 85000, Max 90000, Randy 85000            | cnt=2 -> giữ


### 6.5. Độ phức tạp và điểm vướng của cách 2

Gọi `n` là số nhân viên. Theo đúng ý nghĩa "vòng for lồng nhau", với mỗi `e1` truy vấn con quét các nhân viên khác → khoảng **`O(n²)`** phép so sánh (chính xác hơn là tổng bình phương số người của từng phòng). Bộ nhớ phụ nhỏ (chỉ cần tập lương phân biệt cho một lần đếm, `O(n)` trường hợp xấu nhất).

Database thật có thể chạy nhanh hơn nếu có **index** (chỉ mục) trên `(departmentId, salary)`, nhưng về bản chất ý tưởng vẫn là "mỗi hàng tự đi đếm lại". Ưu điểm lớn: **chạy được trên mọi phiên bản MySQL** (kể cả 5.7 chưa có window function) và rất sát với cách ta nghĩ.

Hai hướng cải tiến:

- **Cách 3** viết lại cùng phép đếm đó bằng `JOIN` + `GROUP BY` thay vì truy vấn con (một kiểu tư duy SQL rất phổ biến, nên học).
- **Cách 4** để database **xếp hạng trong một lần sắp xếp** cho mỗi phòng, thay vì đếm lại cho từng người.

## 7. Cách 3 — Self-join + `GROUP BY` + `HAVING`

### 7.1. Ý tưởng

**Self-join** là `JOIN` một bảng **với chính nó**. Ta ghép mỗi nhân viên `e1` với **mọi** đồng nghiệp `e2` cùng phòng và lương cao hơn:

```sql
FROM Employee e1
LEFT JOIN Employee e2
       ON e2.departmentId = e1.departmentId
      AND e2.salary > e1.salary
```

Sau đó `GROUP BY e1` (mỗi nhân viên một nhóm), đếm `COUNT(DISTINCT e2.salary)` và giữ nhóm có số đếm `< 3` bằng `HAVING`. Đây đúng là phép đếm của cách 2, nhưng thay "truy vấn con cho từng hàng" bằng "tạo hết các cặp rồi gom nhóm".

**Vì sao phải `LEFT JOIN`?** Người lương cao nhất phòng (Max) **không có ai** cao hơn. Với `JOIN` thường, Max không ghép được cặp nào → biến mất khỏi kết quả! `LEFT JOIN` giữ Max lại với `e2` toàn `NULL`, và `COUNT(DISTINCT NULL) = 0` (mục 3.11) → Max có số đếm 0, được giữ. Đẹp!

### 7.2. Nhìn các cặp trung gian trước khi gom nhóm (phòng IT)

Dự đoán: Joe ghép 1 cặp (Max), Max ghép 0 cặp → 1 hàng `NULL`, Janet 4 cặp, Randy 1 cặp, Will 3 cặp → tổng **10 hàng**.

In [20]:
show(db1, """
    SELECT e1.name AS e1_name, e1.salary AS e1_salary,
           e2.name AS e2_name, e2.salary AS e2_salary
    FROM Employee e1
    LEFT JOIN Employee e2
           ON e2.departmentId = e1.departmentId
          AND e2.salary > e1.salary
    WHERE e1.departmentId = 1
    ORDER BY e1.id, e2.salary DESC
""")

+---------+-----------+---------+-----------+
| e1_name | e1_salary | e2_name | e2_salary |
+---------+-----------+---------+-----------+
| Joe     | 85000     | Max     | 90000     |
| Max     | 90000     | null    | null      |
| Janet   | 69000     | Max     | 90000     |
| Janet   | 69000     | Joe     | 85000     |
| Janet   | 69000     | Randy   | 85000     |
| Janet   | 69000     | Will    | 70000     |
| Randy   | 85000     | Max     | 90000     |
| Will    | 70000     | Max     | 90000     |
| Will    | 70000     | Joe     | 85000     |
| Will    | 70000     | Randy   | 85000     |
+---------+-----------+---------+-----------+


Đúng 10 hàng, và Max có một hàng `null` nhờ `LEFT JOIN`. Bây giờ gom theo `e1` và đếm số **mức** phân biệt của `e2.salary`:

| e1 | Các `e2.salary` | `COUNT(DISTINCT e2.salary)` | `HAVING < 3` |
|---|---|:---:|:---:|
| Joe | 90000 | 1 | ✔ |
| Max | null | 0 | ✔ |
| Janet | 90000, 85000, 85000, 70000 | 3 | ✘ |
| Randy | 90000 | 1 | ✔ |
| Will | 90000, 85000, 85000 | 2 | ✔ |

In [21]:
SQL_SELF_JOIN = """
SELECT d.name AS Department, e1.name AS Employee, e1.salary AS Salary
FROM Employee e1
JOIN Department d ON e1.departmentId = d.id
LEFT JOIN Employee e2
       ON e2.departmentId = e1.departmentId
      AND e2.salary > e1.salary
GROUP BY e1.id, d.name, e1.name, e1.salary
HAVING COUNT(DISTINCT e2.salary) < 3
"""

# "trace" nhóm: in số đếm của từng e1 (bỏ HAVING để thấy cả Janet)
show(db1, """
    SELECT e1.name, e1.salary, COUNT(e2.id) AS so_cap, COUNT(DISTINCT e2.salary) AS so_muc_cao_hon
    FROM Employee e1
    LEFT JOIN Employee e2 ON e2.departmentId = e1.departmentId AND e2.salary > e1.salary
    GROUP BY e1.id, e1.name, e1.salary
    ORDER BY e1.id
""")
check_query("Cách 3 - self-join + GROUP BY + HAVING", db1, SQL_SELF_JOIN, EXPECTED_1)

+-------+--------+--------+----------------+
| name  | salary | so_cap | so_muc_cao_hon |
+-------+--------+--------+----------------+
| Joe   | 85000  | 1      | 1              |
| Henry | 80000  | 0      | 0              |
| Sam   | 60000  | 1      | 1              |
| Max   | 90000  | 0      | 0              |
| Janet | 69000  | 4      | 3              |
| Randy | 85000  | 1      | 1              |
| Will  | 70000  | 3      | 2              |
+-------+--------+--------+----------------+
Cách 3 - self-join + GROUP BY + HAVING: PASS (6 hàng)


### 7.3. Ghi chú MySQL và độ phức tạp

- **`GROUP BY` đầy đủ cột.** MySQL bật chế độ `ONLY_FULL_GROUP_BY`: mọi cột trong `SELECT` không nằm trong hàm gộp phải xuất hiện trong `GROUP BY` (hoặc phụ thuộc hàm vào nó). SQLite dễ dãi hơn, cho chạy cả khi thiếu. Để an toàn trên mọi nơi, ta liệt kê đủ `e1.id, d.name, e1.name, e1.salary`. Có `e1.id` trong nhóm là cần thiết: nếu chỉ gom theo `(name, salary)`, hai người khác `id` mà trùng tên và lương ở **khác phòng** sẽ bị dồn thành một (đề chỉ đảm bảo không trùng *cả* tên, lương *và* phòng).
- **Độ phức tạp.** Bảng trung gian sau `LEFT JOIN` có thể có tới cỡ `k²/2` hàng cho một phòng `k` người (xem Janet ghép 4 cặp). Tổng **Time `O(n²)`** và — khác cách 2 — **Space cũng `O(n²)`** cho bảng trung gian ở trường hợp xấu nhất (một phòng rất đông, lương đều khác nhau).

Cả cách 2 và 3 đều "mỗi người tự so với mọi đồng nghiệp". Cái ta thật sự muốn là: **sắp xếp mỗi phòng một lần, rồi gán hạng**. Đó chính là việc của window function.

## 8. Cách 4 — Window function `DENSE_RANK()` trong CTE (cách chính, hiện đại)

### 8.1. Window function là gì? So với `GROUP BY`

`GROUP BY` **gộp** nhiều hàng thành một: sau khi `GROUP BY departmentId`, ta chỉ còn 2 hàng (IT, Sales) và mất thông tin từng nhân viên.

**Window function** (hàm cửa sổ) cũng tính toán trên một nhóm hàng (gọi là "cửa sổ"), nhưng **không gộp**: mỗi hàng vẫn giữ nguyên, chỉ được **gắn thêm một cột** kết quả. Cú pháp:

```sql
TÊN_HÀM(...) OVER (PARTITION BY <cột chia nhóm> ORDER BY <cột sắp xếp>)
```

- `OVER (...)`: dấu hiệu nhận biết đây là window function.
- `PARTITION BY departmentId`: chia hàng thành các "cửa sổ" theo phòng (giống `GROUP BY` nhưng không gộp).
- `ORDER BY salary DESC`: thứ tự các hàng **bên trong mỗi cửa sổ** (quan trọng với các hàm xếp hạng).

Ví dụ: in mỗi nhân viên kèm số người của phòng mình và lương cao nhất phòng mình — vẫn đủ 7 hàng.

In [22]:
show(db1, """
    SELECT name, departmentId, salary,
           COUNT(*)    OVER (PARTITION BY departmentId) AS so_nguoi_phong,
           MAX(salary) OVER (PARTITION BY departmentId) AS max_phong
    FROM Employee
    ORDER BY departmentId, salary DESC
""")

+-------+--------------+--------+----------------+-----------+
| name  | departmentId | salary | so_nguoi_phong | max_phong |
+-------+--------------+--------+----------------+-----------+
| Max   | 1            | 90000  | 5              | 90000     |
| Joe   | 1            | 85000  | 5              | 90000     |
| Randy | 1            | 85000  | 5              | 90000     |
| Will  | 1            | 70000  | 5              | 90000     |
| Janet | 1            | 69000  | 5              | 90000     |
| Henry | 2            | 80000  | 2              | 80000     |
| Sam   | 2            | 60000  | 2              | 80000     |
+-------+--------------+--------+----------------+-----------+


### 8.2. Ba hàm xếp hạng: `ROW_NUMBER` vs `RANK` vs `DENSE_RANK`

Cả ba đánh số các hàng trong mỗi cửa sổ theo `ORDER BY`. Chúng **chỉ khác nhau khi có giá trị trùng** (hòa) — và bài này có hòa (Joe, Randy cùng 85000):

| Hàm | Khi hòa | Sau nhóm hòa | Dãy hạng cho lương IT `90000, 85000, 85000, 70000, 69000` |
|---|---|---|---|
| `ROW_NUMBER()` | hai người hòa vẫn nhận **số khác nhau** (thứ tự giữa họ tùy ý) | tiếp tục +1 | 1, 2, 3, 4, 5 |
| `RANK()` | hai người hòa **cùng hạng** | **nhảy cóc** qua số đã "chiếm" | 1, 2, 2, **4**, 5 |
| `DENSE_RANK()` | hai người hòa **cùng hạng** | **không nhảy cóc**, hạng kế tiếp là +1 | 1, 2, 2, **3**, 4 |

Cách nhớ: `RANK` giống bảng xếp hạng thi chạy (hai người đồng hạng 2 thì không có ai hạng 3, người tiếp theo hạng 4). `DENSE_RANK` ("dày đặc") đếm **mức** giá trị: hạng = thứ tự của **mức lương phân biệt** — đúng y chữ "top three unique salaries".

Chạy cả ba trên Example 1. Với `ROW_NUMBER`, ta thêm `e.id` vào `ORDER BY` để thứ tự giữa người hòa lương cố định (nếu không, database được phép xếp Joe/Randy theo thứ tự nào cũng được).

In [23]:
show(db1, """
    SELECT d.name AS dept, e.name, e.salary,
           ROW_NUMBER() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC, e.id) AS row_num,
           RANK()       OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC)       AS rnk,
           DENSE_RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC)       AS dense_rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
    ORDER BY dept, e.salary DESC, e.id
""")

+-------+-------+--------+---------+-----+-----------+
| dept  | name  | salary | row_num | rnk | dense_rnk |
+-------+-------+--------+---------+-----+-----------+
| IT    | Max   | 90000  | 1       | 1   | 1         |
| IT    | Joe   | 85000  | 2       | 2   | 2         |
| IT    | Randy | 85000  | 3       | 2   | 2         |
| IT    | Will  | 70000  | 4       | 4   | 3         |
| IT    | Janet | 69000  | 5       | 5   | 4         |
| Sales | Henry | 80000  | 1       | 1   | 1         |
| Sales | Sam   | 60000  | 2       | 2   | 2         |
+-------+-------+--------+---------+-----+-----------+


### 8.3. Đọc bảng trên: chỉ `DENSE_RANK` khớp đề

Lọc "hạng `<= 3`" với từng hàm, phòng IT cho:

| Hàm | Ai được giữ ở IT | So với đáp án (Max, Joe, Randy, Will) |
|---|---|---|
| `ROW_NUMBER <= 3` | Max, Joe, Randy | ✘ thiếu Will; và nếu hòa ở vị trí 3 thì giữ ai là **ngẫu nhiên** |
| `RANK <= 3` | Max, Joe, Randy | ✘ thiếu Will vì Will bị đẩy lên hạng 4 |
| `DENSE_RANK <= 3` | Max, Joe, Randy, Will | ✔ |

Lý do sâu xa: `DENSE_RANK` của một người **bằng đúng** `1 + số mức lương phân biệt cao hơn` — chính là đại lượng ta đếm thủ công ở cách 2 và 3 (Will: 2 mức cao hơn → hạng 3). Window function chỉ là cách để database tính nó **hiệu quả** bằng một lần sắp xếp mỗi phòng.

Phòng Sales chỉ có hạng 1 và 2 → `<= 3` giữ cả hai, không cần xử lý gì đặc biệt cho "phòng ít người".

### 8.4. Không lọc window function trong `WHERE` được → dùng CTE

Theo bảng thứ tự thực thi (mục 3.12), `WHERE` chạy **trước** khi window function được tính. Thử viết thẳng sẽ bị lỗi (MySQL cũng báo lỗi tương tự).

Cách sửa: tính hạng ở một bước trước, đặt tên cho kết quả trung gian đó, rồi lọc ở bước sau. Công cụ chuẩn là **CTE** (Common Table Expression — "bảng tạm có tên"):

```sql
WITH ten_bang_tam AS (
    SELECT ...           -- truy vấn tính toán
)
SELECT ... FROM ten_bang_tam WHERE ...;   -- dùng như một bảng bình thường
```

CTE có trong MySQL 8.0+ và SQLite 3.8.3+. Viết kiểu "bảng con trong `FROM`" (`SELECT ... FROM (SELECT ...) AS t WHERE ...`) cũng tương đương.

In [24]:
try:
    run(db1, """SELECT name FROM Employee
                WHERE DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) <= 3""")
except sqlite3.OperationalError as err:
    print("Lỗi như dự đoán:", err)

# CTE mini: bảng tạm "luong_cao" rồi đọc lại như bảng thường
show(db1, """
    WITH luong_cao AS (
        SELECT name, salary FROM Employee WHERE salary >= 80000
    )
    SELECT COUNT(*) AS so_nguoi, MIN(salary) AS thap_nhat FROM luong_cao
""")

Lỗi như dự đoán: misuse of window function DENSE_RANK()
+----------+-----------+
| so_nguoi | thap_nhat |
+----------+-----------+
| 4        | 80000     |
+----------+-----------+


### 8.5. Lời giải chính

Ghép mọi mảnh lại:

1. Trong CTE `ranked`: `JOIN` lấy tên phòng, tính `DENSE_RANK()` theo phòng, lương giảm dần.
2. Ngoài CTE: giữ hàng có `rnk <= 3`, chỉ chọn 3 cột đúng tên đề yêu cầu.

In [25]:
SQL_WINDOW = """
WITH ranked AS (
    SELECT d.name   AS Department,
           e.name   AS Employee,
           e.salary AS Salary,
           DENSE_RANK() OVER (PARTITION BY e.departmentId
                              ORDER BY e.salary DESC) AS rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
SELECT Department, Employee, Salary
FROM ranked
WHERE rnk <= 3
"""

show(db1, SQL_WINDOW)
check_query("Cách 4 - DENSE_RANK trong CTE", db1, SQL_WINDOW, EXPECTED_1)

+------------+----------+--------+
| Department | Employee | Salary |
+------------+----------+--------+
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |
+------------+----------+--------+
Cách 4 - DENSE_RANK trong CTE: PASS (6 hàng)


### 8.6. Trace cách 4: nhìn vào bên trong CTE

Bảng `ranked` (trước khi lọc) của Example 1, kèm quyết định:

| Department | Employee | Salary | rnk | `rnk <= 3`? |
|---|---|---:|:---:|:---:|
| IT | Max | 90000 | 1 | giữ |
| IT | Joe | 85000 | 2 | giữ |
| IT | Randy | 85000 | 2 | giữ |
| IT | Will | 70000 | 3 | giữ |
| IT | Janet | 69000 | 4 | **loại** |
| Sales | Henry | 80000 | 1 | giữ |
| Sales | Sam | 60000 | 2 | giữ |

Ta in đúng bảng này bằng SQL, dùng thêm biểu thức **`CASE WHEN ... THEN ... ELSE ... END`** — "if/else" của SQL — để hiện cột quyết định.

In [26]:
show(db1, """
    WITH ranked AS (
        SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
               DENSE_RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS rnk
        FROM Employee e
        JOIN Department d ON e.departmentId = d.id
    )
    SELECT Department, Employee, Salary, rnk,
           CASE WHEN rnk <= 3 THEN 'giữ' ELSE 'LOẠI' END AS quyet_dinh
    FROM ranked
    ORDER BY Department, rnk, Employee
""")

+------------+----------+--------+-----+------------+
| Department | Employee | Salary | rnk | quyet_dinh |
+------------+----------+--------+-----+------------+
| IT         | Max      | 90000  | 1   | giữ        |
| IT         | Joe      | 85000  | 2   | giữ        |
| IT         | Randy    | 85000  | 2   | giữ        |
| IT         | Will     | 70000  | 3   | giữ        |
| IT         | Janet    | 69000  | 4   | LOẠI       |
| Sales      | Henry    | 80000  | 1   | giữ        |
| Sales      | Sam      | 60000  | 2   | giữ        |
+------------+----------+--------+-----+------------+


### 8.7. Độ phức tạp cách 4

Với `n` nhân viên: database cần sắp các hàng theo `(departmentId, salary DESC)` — **`O(n log n)`** — rồi quét một lượt để gán hạng `O(n)`. `JOIN` với `Department` (thường qua index trên khóa chính) không làm tăng bậc. **Time `O(n log n)`**, **Space `O(n)`** cho bảng `ranked`.

So với `O(n²)` của cách 2–3, đây là bước nhảy lớn khi phòng ban đông người. Ngoài ra câu truy vấn đọc như lời đề: "xếp hạng dày đặc theo phòng, lấy hạng ≤ 3". Window function có trong MySQL 8.0+, PostgreSQL, SQL Server, Oracle, SQLite 3.25+ → đây là **cách nên nộp**.

## 9. Cách 5 (tùy chọn) — Pandas

LeetCode có tab **Pandas** cho bài này. Pandas là thư viện bảng dữ liệu của Python (không thuộc thư viện chuẩn — cell dưới tự bỏ qua nếu máy bạn chưa cài). Bảng đối chiếu khái niệm:

| SQL | Pandas |
|---|---|
| bảng | `DataFrame` |
| `JOIN ... ON e.departmentId = d.id` | `employee.merge(department, left_on="departmentId", right_on="id")` |
| `DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC)` | `df.groupby("departmentId")["salary"].rank(method="dense", ascending=False)` |
| `WHERE rnk <= 3` | `df[df["rnk"] <= 3]` |
| `SELECT a AS X` | `df[["a"]].rename(columns={"a": "X"})` |

Lưu ý: cả hai bảng đều có cột `id` và `name`, nên `merge` tự thêm hậu tố; ta chỉ định `suffixes=("_emp", "_dept")` để tên rõ nghĩa (`name_emp`, `name_dept`). `method="dense"` chính là `DENSE_RANK`; `rank` của pandas trả về số thực (`1.0`, `2.0`...), so sánh `<= 3` vẫn đúng.

In [27]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("Chưa cài pandas -> bỏ qua cách 5 (không ảnh hưởng phần còn lại).")

if pd is not None:
    def top_three_salaries(employee: "pd.DataFrame", department: "pd.DataFrame") -> "pd.DataFrame":
        df = employee.merge(department, left_on="departmentId", right_on="id",
                            suffixes=("_emp", "_dept"))
        df["rnk"] = df.groupby("departmentId")["salary"].rank(method="dense", ascending=False)
        top = df[df["rnk"] <= 3]
        return top[["name_dept", "name_emp", "salary"]].rename(
            columns={"name_dept": "Department", "name_emp": "Employee", "salary": "Salary"})

    def run_pandas(employees, departments):
        emp_df = pd.DataFrame(employees, columns=["id", "name", "salary", "departmentId"])
        dep_df = pd.DataFrame(departments, columns=["id", "name"])
        out = top_three_salaries(emp_df, dep_df)
        assert list(out.columns) == RESULT_COLS
        return [(a, b, int(c)) for a, b, c in out.itertuples(index=False)]

    print(top_three_salaries(pd.DataFrame(EMPLOYEES_1, columns=["id", "name", "salary", "departmentId"]),
                             pd.DataFrame(DEPARTMENTS_1, columns=["id", "name"])))
    assert sorted(run_pandas(EMPLOYEES_1, DEPARTMENTS_1)) == sorted(EXPECTED_1)
    print("Cách 5 - pandas: PASS")

  Department Employee  Salary
0         IT      Joe   85000
1      Sales    Henry   80000
2      Sales      Sam   60000
3         IT      Max   90000
5         IT    Randy   85000
6         IT     Will   70000
Cách 5 - pandas: PASS


## 10. Code nộp LeetCode

Bài Database **không có** `class Solution`. Trên LeetCode chọn ngôn ngữ **MySQL** và dán nguyên câu truy vấn. Câu dưới đây dùng cú pháp chung của MySQL 8 và SQLite, nên ta chạy được ngay để kiểm tra.

```sql
WITH ranked AS (
    SELECT d.name   AS Department,
           e.name   AS Employee,
           e.salary AS Salary,
           DENSE_RANK() OVER (PARTITION BY e.departmentId
                              ORDER BY e.salary DESC) AS rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
SELECT Department, Employee, Salary
FROM ranked
WHERE rnk <= 3;
```

Nếu gặp môi trường MySQL cũ (5.7, chưa có `WITH` và window function), nộp **cách 2** — correlated subquery — vẫn được chấp nhận. Dấu `;` cuối câu là tùy chọn trên LeetCode.

Một khác biệt dialect cần biết: SQLite không phân biệt hoa/thường tên bảng, còn MySQL trên Linux **có** phân biệt → luôn viết đúng `Employee`, `Department` như đề.

In [28]:
LEETCODE_MYSQL = """
WITH ranked AS (
    SELECT d.name   AS Department,
           e.name   AS Employee,
           e.salary AS Salary,
           DENSE_RANK() OVER (PARTITION BY e.departmentId
                              ORDER BY e.salary DESC) AS rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
)
SELECT Department, Employee, Salary
FROM ranked
WHERE rnk <= 3;
"""

LEETCODE_MYSQL_NO_WINDOW = SQL_CORRELATED.strip() + ";"   # bản dự phòng cho MySQL 5.7

check_query("LeetCode - bản chính (DENSE_RANK)", db1, LEETCODE_MYSQL, EXPECTED_1)
check_query("LeetCode - bản dự phòng (correlated)", db1, LEETCODE_MYSQL_NO_WINDOW, EXPECTED_1)

LeetCode - bản chính (DENSE_RANK): PASS (6 hàng)
LeetCode - bản dự phòng (correlated): PASS (6 hàng)


## 11. Kiểm thử: edge case và đối chiếu chéo các cách

### 11.1. Bộ dữ liệu edge case (tự tạo, không phải của LeetCode)

Example 1 chưa thử hết các tình huống dễ sai. Ta tự dựng một công ty 6 phòng, mỗi phòng nhắm vào một cái bẫy:

| Phòng | Dữ liệu lương | Bẫy kiểm tra | Kỳ vọng |
|---|---|---|---|
| IT | 85k, 90k, 69k, 85k, 70k (như ví dụ) | hòa ở hạng 2 | Max, Joe, Randy, Will |
| Sales | 80k, 60k | chỉ 2 mức lương | cả 2 |
| HR | 50k | chỉ 1 người | Lan |
| Legal | (không ai) | phòng trống | không có hàng nào |
| Ops | 40k, 40k, 40k | mọi người cùng lương | cả 3 |
| RnD | 90k, 90k, 80k, 70k, 70k, 60k | hòa ở hạng 1 **và** hạng 3 | 5 người, trừ Vy (60k) |

Tổng kỳ vọng: 4 + 2 + 1 + 0 + 3 + 5 = **15 hàng**.

In [29]:
EMPLOYEES_EDGE = EMPLOYEES_1 + [
    (8, "Lan", 50000, 3),
    (9, "Minh", 40000, 5), (10, "Nam", 40000, 5), (11, "Oanh", 40000, 5),
    (12, "Phong", 90000, 6), (13, "Quan", 90000, 6), (14, "Son", 80000, 6),
    (15, "Tam", 70000, 6), (16, "Uyen", 70000, 6), (17, "Vy", 60000, 6),
]
DEPARTMENTS_EDGE = DEPARTMENTS_1 + [(3, "HR"), (4, "Legal"), (5, "Ops"), (6, "RnD")]
EXPECTED_EDGE = EXPECTED_1 + [
    ("HR", "Lan", 50000),
    ("Ops", "Minh", 40000), ("Ops", "Nam", 40000), ("Ops", "Oanh", 40000),
    ("RnD", "Phong", 90000), ("RnD", "Quan", 90000), ("RnD", "Son", 80000),
    ("RnD", "Tam", 70000), ("RnD", "Uyen", 70000),
]
assert len(EXPECTED_EDGE) == 15

db_edge = make_db(EMPLOYEES_EDGE, DEPARTMENTS_EDGE)
assert sorted(top_three_python(EMPLOYEES_EDGE, DEPARTMENTS_EDGE)) == sorted(EXPECTED_EDGE)
print("Cách 1 - python: PASS")
for name, sql in [("Cách 2 - correlated", SQL_CORRELATED),
                  ("Cách 3 - self-join", SQL_SELF_JOIN),
                  ("Cách 4 - window", SQL_WINDOW),
                  ("LeetCode MySQL", LEETCODE_MYSQL)]:
    check_query(name, db_edge, sql, EXPECTED_EDGE)
if pd is not None:
    assert sorted(run_pandas(EMPLOYEES_EDGE, DEPARTMENTS_EDGE)) == sorted(EXPECTED_EDGE)
    print("Cách 5 - pandas: PASS")

Cách 1 - python: PASS
Cách 2 - correlated: PASS (15 hàng)
Cách 3 - self-join: PASS (15 hàng)
Cách 4 - window: PASS (15 hàng)
LeetCode MySQL: PASS (15 hàng)
Cách 5 - pandas: PASS


### 11.2. Đối chiếu chéo trên dữ liệu ngẫu nhiên

Ta sinh 300 công ty nhỏ ngẫu nhiên (`random.seed(185)` để lần nào chạy cũng như nhau): 1–4 phòng, 0–12 nhân viên, lương chỉ lấy từ 5 mức để **ép xảy ra hòa** thật nhiều. Cách 1 (Python thuần, dễ tin nhất) đóng vai "trọng tài"; mọi cách khác phải cho **cùng tập kết quả** (sau khi sắp xếp).

Tên nhân viên là `E1, E2, ...` (không trùng) nên ràng buộc "không trùng cả tên, lương và phòng" luôn được thỏa.

In [30]:
import random

def random_company(rng):
    n_dept = rng.randint(1, 4)
    departments = [(i, f"D{i}") for i in range(1, n_dept + 1)]
    employees = [(i, f"E{i}", rng.choice([1000, 2000, 3000, 4000, 5000]), rng.randint(1, n_dept))
                 for i in range(1, rng.randint(0, 12) + 1)]
    return employees, departments


random.seed(185)
rng = random.Random(185)
ALL_SQL = {"correlated": SQL_CORRELATED, "self_join": SQL_SELF_JOIN, "window": SQL_WINDOW}
total_rows = 0
for trial in range(300):
    emps, deps = random_company(rng)
    expected = sorted(top_three_python(emps, deps))
    db = make_db(emps, deps)
    for name, sql in ALL_SQL.items():
        got = sorted(run(db, sql)[1])
        assert got == expected, (trial, name, emps, got, expected)
    if pd is not None:
        assert sorted(run_pandas(emps, deps)) == expected, (trial, "pandas")
    total_rows += len(expected)
    db.close()

print(f"300 công ty ngẫu nhiên: mọi cách đều khớp trọng tài Python ({total_rows} hàng kết quả tổng cộng)")

300 công ty ngẫu nhiên: mọi cách đều khớp trọng tài Python (1659 hàng kết quả tổng cộng)


## 12. Vì sao đúng? Độ phức tạp và so sánh các cách

### 12.1. Invariant (bất biến) nối mọi cách lại với nhau

Xét một phòng `D` và một nhân viên `e` thuộc `D`. Gọi `S` là tập **mức lương phân biệt** của `D`, sắp giảm dần `s₁ > s₂ > s₃ > ...`. Định nghĩa:

> `higher(e)` = số phần tử của `S` **lớn hơn** `e.salary`.

- Nếu `e.salary = sⱼ` thì đúng có `j − 1` mức lớn hơn nó (là `s₁, ..., sⱼ₋₁`), nên `higher(e) = j − 1`.
- `e` thuộc "top 3 mức phân biệt" ⇔ `j ≤ 3` ⇔ **`higher(e) < 3`**. Đây là điều kiện của cách 2 (`COUNT(DISTINCT e2.salary)` trong truy vấn con) và cách 3 (`COUNT(DISTINCT e2.salary)` sau `LEFT JOIN`, với `NULL` không được đếm nên người cao nhất có 0).
- `DENSE_RANK()` theo `salary DESC` trong `PARTITION BY departmentId` gán cho `e` đúng số thứ tự `j` của mức lương của nó (người hòa cùng `j`, không nhảy cóc) ⇒ **`DENSE_RANK(e) = higher(e) + 1`**, nên `rnk <= 3` ⇔ `higher(e) < 3`. Đây là cách 4 và 5.

**Không bỏ sót, không lấy thừa:** điều kiện được kiểm tra cho **từng** nhân viên một cách độc lập; mọi phòng — kể cả phòng chỉ có 1–2 mức lương — đều có `higher(e) ≤ |S| − 1 < 3` cho tất cả, nên không ai bị loại oan. Phòng không có nhân viên không sinh ra hàng nào (đề không yêu cầu). `JOIN Department` không làm mất ai vì `departmentId` là khóa ngoại hợp lệ.

Cell dưới kiểm chứng đẳng thức `DENSE_RANK = 1 + higher` trên dữ liệu edge case bằng một câu SQL duy nhất.

In [31]:
cols, rows = run(db_edge, """
    SELECT e1.id,
           DENSE_RANK() OVER (PARTITION BY e1.departmentId ORDER BY e1.salary DESC) AS dr,
           1 + (SELECT COUNT(DISTINCT e2.salary) FROM Employee e2
                WHERE e2.departmentId = e1.departmentId AND e2.salary > e1.salary) AS one_plus_higher
    FROM Employee e1
""")
mismatch = [r for r in rows if r[1] != r[2]]
print(f"Đã so {len(rows)} nhân viên, số chỗ lệch: {len(mismatch)}")
assert not mismatch

Đã so 17 nhân viên, số chỗ lệch: 0


### 12.2. Bảng so sánh

`n` = số nhân viên, `k` = số người của phòng đông nhất.

| Cách | Ý tưởng | Time | Space phụ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1. Python thuần | dict phòng → set lương → top 3 | `O(n log n)` | `O(n)` | dễ hiểu, làm trọng tài test | không phải SQL, phải kéo hết dữ liệu về |
| 2. Correlated subquery | mỗi người đếm số mức cao hơn `< 3` | `O(n·k)` ≤ `O(n²)` | nhỏ | chạy mọi bản MySQL, sát cách nghĩ | chậm khi phòng đông |
| 3. Self-join + GROUP BY + HAVING | tạo cặp (mình, người cao hơn) rồi đếm | `O(n·k)` ≤ `O(n²)` | `O(n·k)` bảng trung gian | luyện JOIN/GROUP BY/HAVING/LEFT JOIN | tốn bộ nhớ, dễ quên `LEFT` |
| 4. **`DENSE_RANK` + CTE** | sắp mỗi phòng một lần, gán hạng dày đặc | `O(n log n)` | `O(n)` | ngắn, đọc như đề, nhanh — **nên nộp** | cần MySQL 8+ |
| 5. Pandas | `merge` + `groupby().rank("dense")` | `O(n log n)` | `O(n)` | dùng cho tab Pandas | thư viện ngoài |

## 13. Lỗi người mới hay gặp (chạy thật trên bộ edge case)

| # | Code sai | Vì sao sai | Sửa |
|:---:|---|---|---|
| 1 | `RANK() ... <= 3` | hòa làm `RANK` nhảy cóc: IT mất Will (hạng 4), RnD mất Tam, Uyen (hạng 4) | `DENSE_RANK()` |
| 2 | `ROW_NUMBER() ... <= 3` | mỗi phòng tối đa 3 người: IT mất Will, RnD mất Tam, Uyen; khi hòa ở vị trí 3 thì giữ ai là tùy ý | `DENSE_RANK()` |
| 3 | `ORDER BY salary DESC LIMIT 3` | `LIMIT` cắt **toàn bộ** kết quả, không theo phòng, không theo mức | xếp hạng theo `PARTITION BY` |
| 4 | `COUNT(e2.salary)` (thiếu `DISTINCT`) | đếm **người** thay vì **mức**: Will thấy 3 người cao hơn → loại oan (RnD cũng mất Tam, Uyen) | `COUNT(DISTINCT e2.salary)` |
| 5 | `... <= 3` trong cách đếm | người ở mức thứ 4 có đúng 3 mức cao hơn → lấy thừa (Janet, Vy) | `< 3` |
| 6 | Self-join bằng `JOIN` thường | người lương cao nhất không có cặp → biến mất (Max, Henry, Lan...) | `LEFT JOIN` |
| 7 | "lấy mức lương thứ 3 rồi `salary >= mức đó`" | phòng có < 3 mức → truy vấn con trả `NULL`, `salary >= NULL` là `NULL` → **mất cả phòng** (Sales, HR, Ops) | `COALESCE(..., 0)` hoặc dùng cách đếm/hạng |
| 8 | Quên `AS Department/Employee/Salary` | tên cột kết quả sai → LeetCode chấm sai dù dữ liệu đúng | đặt alias đúng chữ hoa/thường |

Cell dưới chạy các phiên bản sai và in **ai bị thiếu / ai bị thừa** so với đáp án.

In [32]:
def diff_report(name, sql, db=db_edge, expected=EXPECTED_EDGE):
    got = run(db, sql)[1]
    missing = sorted(set(expected) - set(got))
    extra = sorted(set(got) - set(expected))
    status = "ĐÚNG" if not missing and not extra and len(got) == len(expected) else "SAI"
    print(f"[{status}] {name}: {len(got)} hàng | thiếu {[m[1] for m in missing]} | thừa {[x[1] for x in extra]}")


WINDOW_TEMPLATE = """
WITH ranked AS (
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
           {fn}() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC{tie}) AS rnk
    FROM Employee e JOIN Department d ON e.departmentId = d.id
)
SELECT Department, Employee, Salary FROM ranked WHERE rnk <= 3
"""
diff_report("Lỗi 1 - RANK", WINDOW_TEMPLATE.format(fn="RANK", tie=""))
diff_report("Lỗi 2 - ROW_NUMBER", WINDOW_TEMPLATE.format(fn="ROW_NUMBER", tie=", e.id"))
diff_report("Lỗi 3 - LIMIT 3", """
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
    FROM Employee e JOIN Department d ON e.departmentId = d.id
    ORDER BY e.salary DESC LIMIT 3""")
diff_report("Đúng   - DENSE_RANK", WINDOW_TEMPLATE.format(fn="DENSE_RANK", tie=""))

[SAI] Lỗi 1 - RANK: 12 hàng | thiếu ['Will', 'Tam', 'Uyen'] | thừa []
[SAI] Lỗi 2 - ROW_NUMBER: 12 hàng | thiếu ['Will', 'Tam', 'Uyen'] | thừa []
[SAI] Lỗi 3 - LIMIT 3: 3 hàng | thiếu ['Lan', 'Joe', 'Randy', 'Will', 'Minh', 'Nam', 'Oanh', 'Son', 'Tam', 'Uyen', 'Henry', 'Sam'] | thừa []
[ĐÚNG] Đúng   - DENSE_RANK: 15 hàng | thiếu [] | thừa []


In [33]:
diff_report("Lỗi 4 - COUNT thiếu DISTINCT", SQL_CORRELATED.replace("COUNT(DISTINCT e2.salary)", "COUNT(e2.salary)"))
diff_report("Lỗi 5 - dùng <= 3", SQL_CORRELATED.replace(") < 3", ") <= 3"))
diff_report("Lỗi 6 - self-join bằng JOIN thường", SQL_SELF_JOIN.replace("LEFT JOIN", "JOIN"))

THIRD_SALARY = """
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
WHERE e.salary >= {third}
"""
SUB = """(SELECT DISTINCT x.salary FROM Employee x WHERE x.departmentId = e.departmentId
          ORDER BY x.salary DESC LIMIT 1 OFFSET 2)"""
diff_report("Lỗi 7 - so với mức thứ 3 (NULL)", THIRD_SALARY.format(third=SUB))
diff_report("Sửa 7 - COALESCE(mức thứ 3, 0)", THIRD_SALARY.format(third=f"COALESCE({SUB}, 0)"))

print("Lỗi 8 - tên cột khi quên alias:", run(db_edge, "SELECT d.name, e.name, e.salary FROM Employee e "
                                              "JOIN Department d ON e.departmentId = d.id LIMIT 1")[0])

[SAI] Lỗi 4 - COUNT thiếu DISTINCT: 12 hàng | thiếu ['Will', 'Tam', 'Uyen'] | thừa []
[SAI] Lỗi 5 - dùng <= 3: 17 hàng | thiếu [] | thừa ['Janet', 'Vy']
[SAI] Lỗi 6 - self-join bằng JOIN thường: 7 hàng | thiếu ['Lan', 'Max', 'Minh', 'Nam', 'Oanh', 'Phong', 'Quan', 'Henry'] | thừa []
[SAI] Lỗi 7 - so với mức thứ 3 (NULL): 9 hàng | thiếu ['Lan', 'Minh', 'Nam', 'Oanh', 'Henry', 'Sam'] | thừa []
[ĐÚNG] Sửa 7 - COALESCE(mức thứ 3, 0): 15 hàng | thiếu [] | thừa []
Lỗi 8 - tên cột khi quên alias: ['name', 'name', 'salary']


Nhận xét: lỗi 7 là cái bẫy "**quên phòng có ít hơn 3 người**" kinh điển — trông rất hợp lý, chạy đúng với IT và RnD, nhưng âm thầm xóa sổ Sales, HR và Ops vì so sánh với `NULL`. `COALESCE(a, b)` trả về `a` nếu `a` khác `NULL`, ngược lại trả `b`; thay mức thứ 3 "không tồn tại" bằng `0` thì mọi người trong phòng nhỏ đều đạt (lương trong bài là số dương). Dù vậy, cách đếm/hạng vẫn gọn và ít bẫy hơn.

## 14. Mẹo nhận dạng pattern và bài liên quan

**Dấu hiệu nhận dạng "Top-N mỗi nhóm" (top-N per group):**

- Đề có cụm "**in each** department / for each user / per category" + "**top k** / highest / most recent k".
- Kết quả giữ **chi tiết từng hàng** (tên nhân viên), không chỉ một con số mỗi nhóm → `GROUP BY` thuần không đủ, nghĩ ngay tới **window function** với `PARTITION BY <nhóm>`.
- Chọn hàm xếp hạng theo **cách xử lý hòa** của đề:
  - "top k **giá trị phân biệt**", người hòa cùng được tính → `DENSE_RANK`.
  - "xếp hạng kiểu thi đấu, đồng hạng thì nhảy cóc" → `RANK`.
  - "đúng k **hàng**, hòa thì chọn một" (vd: k đơn hàng gần nhất) → `ROW_NUMBER` (+ tiêu chí phụ để cố định).
- Không có window function (MySQL cũ)? → **correlated subquery đếm số giá trị phân biệt lớn hơn `< k`**.

**Bài liên quan trên LeetCode:**

| Bài | Nội dung | Kỹ thuật |
|---|---|---|
| 176. Second Highest Salary | mức lương cao thứ 2 toàn công ty, không có thì `null` | `DISTINCT` + `LIMIT 1 OFFSET 1` bọc trong scalar subquery |
| 177. Nth Highest Salary | mức lương cao thứ N (viết `FUNCTION` trong MySQL) | như 176 với `OFFSET N-1`, hoặc `DENSE_RANK` |
| 178. Rank Scores | xếp hạng điểm, hòa cùng hạng, không nhảy cóc | `DENSE_RANK` trên toàn bảng |
| 184. Department Highest Salary | người lương cao nhất mỗi phòng (có thể nhiều người) | `DENSE_RANK = 1`, hoặc so với `MAX` theo phòng |
| 185. (bài này) | top 3 mức lương phân biệt mỗi phòng | `DENSE_RANK <= 3` |

Bài 176, 177, 184 sẽ là bài tập ngay bên dưới.

## 15. Bài tập vận dụng (tăng dần)

Tự làm trước, **chưa xem** mục 16–17. Các cell TODO chỉ định nghĩa hàm/dữ liệu, assert được comment sẵn — làm xong thì mở comment để tự chấm.

### Bài 1 — Đoán output (không chạy code)

Một phòng có lương (đã sắp giảm dần): `100, 100, 90, 80, 80, 70`.

a) Viết dãy giá trị của `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()` (theo `salary DESC`; với `ROW_NUMBER` coi người hòa xếp theo thứ tự đã cho).
b) Lọc `<= 3` theo từng hàm thì giữ được bao nhiêu người? Hàm nào cho đúng yêu cầu bài 185?

### Bài 2 — Tính tay correlated subquery

Phòng `Ops` có: An 3000, Bình 5000, Chi 4000, Dũng 5000, Em 2000, Giang 4000 (theo thứ tự `id`). Với **từng người**, tính `COUNT(DISTINCT e2.salary)` của những đồng nghiệp lương **cao hơn**, rồi cho biết ai nằm trong kết quả bài 185. Lập bảng như mục 6.4.

### Bài 3 — Điền TODO: LeetCode 176 Second Highest Salary

Bảng `Employee(id INT, salary INT)` (`id` là khóa chính). Viết câu SQL trả về **mức lương phân biệt cao thứ 2**, trong một cột tên `SecondHighestSalary`. Nếu không có mức thứ 2, trả về `null` (vẫn phải có **một hàng** chứa `null`, không được trả bảng rỗng).

Ví dụ của đề: lương `100, 200, 300` → `200`; lương chỉ có `100` → `null`.

Hàm của bạn chỉ cần **trả về chuỗi SQL**.

In [34]:
def make_salary_db(salaries):
    """Bảng Employee(id, salary) cho bài 176/177."""
    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE Employee (id INT PRIMARY KEY, salary INT)")
    db.executemany("INSERT INTO Employee VALUES (?, ?)", list(enumerate(salaries, start=1)))
    return db


def exercise_second_highest_sql() -> str:
    # TODO 1: lấy các mức lương PHÂN BIỆT, sắp giảm dần
    # TODO 2: bỏ qua 1 mức đầu, lấy 1 mức (LIMIT ... OFFSET ...)
    # TODO 3: bọc trong (SELECT ...) để không có kết quả thì ra NULL, đặt tên cột SecondHighestSalary
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# for sal, exp in [([100, 200, 300], 200), ([100], None), ([100, 100], None), ([300, 300, 200], 200)]:
#     assert run(make_salary_db(sal), exercise_second_highest_sql()) == (["SecondHighestSalary"], [(exp,)])

### Bài 4 — Tự viết: LeetCode 177 Nth Highest Salary

Cùng bảng `Employee(id, salary)`. Trả về mức lương phân biệt cao thứ `n`, hoặc `None` (`null`) nếu không tồn tại. Ví dụ đề: lương `100, 200, 300`, `n = 2` → `200`; lương `100`, `n = 2` → `null`.

Trên LeetCode bài này phải viết dưới dạng **hàm lưu trữ** của MySQL (`CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT BEGIN ... END`). SQLite không có `CREATE FUNCTION` kiểu đó, nên ở đây ta mô phỏng: viết hàm Python nhận `db` và `n`, chạy một câu SQL có **placeholder `?`** rồi trả về con số. Hãy thử làm **hai phiên bản**: một bằng `LIMIT/OFFSET`, một bằng `DENSE_RANK`.

In [35]:
def exercise_nth_highest(db, n: int):
    # TODO: chạy SQL với tham số (n - 1) cho OFFSET, hoặc lọc DENSE_RANK() = n
    # Gợi ý dạng gọi: db.execute("SELECT ... OFFSET ?", (n - 1,)).fetchone()[0]
    pass


# Tự làm xong rồi mới mở test:
# db_177 = make_salary_db([100, 200, 300])
# assert exercise_nth_highest(db_177, 1) == 300
# assert exercise_nth_highest(db_177, 2) == 200
# assert exercise_nth_highest(db_177, 4) is None
# assert exercise_nth_highest(make_salary_db([100]), 2) is None
# assert exercise_nth_highest(make_salary_db([200, 200, 100]), 2) == 100

### Bài 5 — Biến thể: LeetCode 184 Department Highest Salary

Cùng schema với bài 185. Tìm nhân viên có lương **cao nhất** trong mỗi phòng (hòa thì lấy hết). Cột kết quả: `Department`, `Employee`, `Salary`, thứ tự bất kỳ.

Ví dụ của đề 184 — `Employee`: Joe 70000 (phòng 1), Jim 90000 (1), Henry 80000 (2), Sam 60000 (2), Max 90000 (1); `Department`: 1 IT, 2 Sales. Output: (IT, Jim, 90000), (Sales, Henry, 80000), (IT, Max, 90000).

Hãy viết **hai** câu: một sửa từ lời giải chính của bài này (chỉ đổi 1 chỗ!), một **không** dùng window function.

In [36]:
EMPLOYEES_184 = [(1, "Joe", 70000, 1), (2, "Jim", 90000, 1), (3, "Henry", 80000, 2),
                 (4, "Sam", 60000, 2), (5, "Max", 90000, 1)]
DEPARTMENTS_184 = [(1, "IT"), (2, "Sales")]
EXPECTED_184 = [("IT", "Jim", 90000), ("Sales", "Henry", 80000), ("IT", "Max", 90000)]


def exercise_dept_highest_window_sql() -> str:
    # TODO: CTE + DENSE_RANK, nhưng giữ hạng nào?
    raise NotImplementedError


def exercise_dept_highest_no_window_sql() -> str:
    # TODO: so e.salary với MAX(salary) của CHÍNH phòng đó (correlated subquery hoặc JOIN bảng gộp)
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# db_184 = make_db(EMPLOYEES_184, DEPARTMENTS_184)
# check_query("184 window", db_184, exercise_dept_highest_window_sql(), EXPECTED_184)
# check_query("184 no window", db_184, exercise_dept_highest_no_window_sql(), EXPECTED_184)
# check_query("184 window (edge)", db_edge, exercise_dept_highest_window_sql(),
#             [r for r in EXPECTED_EDGE if r[2] == max(x[2] for x in EXPECTED_EDGE if x[0] == r[0])])

### Bài 6 — Mini challenge: thêm cột hạng, **không** dùng window function

Viết lại kết quả bài 185 nhưng thêm cột thứ tư `SalaryRank` (1, 2 hoặc 3 — hạng của mức lương trong phòng), và **sắp xếp** theo `Department` tăng dần, rồi `SalaryRank` tăng dần, rồi `Employee` tăng dần. Ràng buộc: **không** được dùng `DENSE_RANK`/`RANK`/`ROW_NUMBER` (giả sử bạn đang ở MySQL 5.7).

Lần này thứ tự **có** ý nghĩa, nên test so sánh nguyên danh sách, không sắp xếp lại.

Gợi ý suy nghĩ: mục 12.1 đã chứng minh một đẳng thức cho bạn biết `SalaryRank` bằng gì. Và chú ý: MySQL **không** cho dùng alias của `SELECT` trong `WHERE` (SQLite thì cho — đừng dựa vào điều đó).

In [37]:
def exercise_rank_no_window_sql() -> str:
    # TODO 1: SalaryRank = 1 + (correlated subquery đếm mức lương phân biệt cao hơn)
    # TODO 2: bọc trong CTE / bảng con để lọc SalaryRank <= 3 (đừng lọc alias trực tiếp trong WHERE)
    # TODO 3: ORDER BY Department, SalaryRank, Employee
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# expected_6 = [("HR", "Lan", 50000, 1),
#               ("IT", "Max", 90000, 1), ("IT", "Joe", 85000, 2), ("IT", "Randy", 85000, 2), ("IT", "Will", 70000, 3),
#               ("Ops", "Minh", 40000, 1), ("Ops", "Nam", 40000, 1), ("Ops", "Oanh", 40000, 1),
#               ("RnD", "Phong", 90000, 1), ("RnD", "Quan", 90000, 1), ("RnD", "Son", 80000, 2),
#               ("RnD", "Tam", 70000, 3), ("RnD", "Uyen", 70000, 3),
#               ("Sales", "Henry", 80000, 1), ("Sales", "Sam", 60000, 2)]
# cols, rows = run(db_edge, exercise_rank_no_window_sql())
# assert cols == ["Department", "Employee", "Salary", "SalaryRank"] and rows == expected_6

## 16. Gợi ý theo mức

**Bài 1.**
- Mức 1: `ROW_NUMBER` không bao giờ cho hai hàng cùng số.
- Mức 2: `RANK` sau hai người đồng hạng 1 sẽ nhảy tới hạng 3; `DENSE_RANK` thì tới hạng 2.
- Mức 3: đếm xem hạng ≤ 3 dừng ở người thứ mấy với mỗi dãy.

**Bài 2.**
- Mức 1: chỉ có 4 mức lương phân biệt: 5000, 4000, 3000, 2000.
- Mức 2: Bình và Dũng (5000) không có ai cao hơn; người 4000 chỉ thấy mức 5000.
- Mức 3: Em (2000) thấy 3 mức cao hơn → loại; còn lại giữ.

**Bài 3.**
- Mức 1: "cao thứ 2" = bỏ qua 1 mức cao nhất trong danh sách mức **phân biệt**.
- Mức 2: `SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1` — nhưng nếu không có mức thứ 2, câu này trả bảng **rỗng**, chứ không phải `null`.
- Mức 3: scalar subquery không có hàng nào sẽ cho `NULL`: `SELECT ( ...câu trên... ) AS SecondHighestSalary`. Cách khác: `SELECT MAX(salary) ... WHERE salary < (SELECT MAX(salary) FROM Employee)` (hàm gộp trên tập rỗng cũng cho `NULL`).

**Bài 4.**
- Mức 1: tổng quát bài 3: `OFFSET n - 1`.
- Mức 2: truyền tham số qua `db.execute(sql, (n - 1,))`; `fetchone()` lấy hàng đầu tiên, `[0]` lấy cột đầu.
- Mức 3: bản `DENSE_RANK`: `SELECT MAX(salary) FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS r FROM Employee) t WHERE r = ?`. (Trong MySQL, `LIMIT/OFFSET` không nhận biểu thức `N-1`, nên trong `FUNCTION` phải `SET N = N - 1;` trước.)

**Bài 5.**
- Mức 1: "cao nhất" = mức lương phân biệt hạng 1.
- Mức 2: bản window: đổi `rnk <= 3` thành `rnk = 1`.
- Mức 3: bản không window: `WHERE e.salary = (SELECT MAX(e2.salary) FROM Employee e2 WHERE e2.departmentId = e.departmentId)`.

**Bài 6.**
- Mức 1: `SalaryRank = 1 + higher(e)` (mục 12.1).
- Mức 2: tính cột `SalaryRank` trong `WITH t AS (SELECT ..., 1 + (SELECT COUNT(DISTINCT ...) ...) AS SalaryRank FROM ...)`.
- Mức 3: `SELECT * FROM t WHERE SalaryRank <= 3 ORDER BY Department, SalaryRank, Employee`.

## 17. Đáp án tham khảo

### Bài 1

| Lương | 100 | 100 | 90 | 80 | 80 | 70 |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| `ROW_NUMBER` | 1 | 2 | 3 | 4 | 5 | 6 |
| `RANK` | 1 | 1 | 3 | 4 | 4 | 6 |
| `DENSE_RANK` | 1 | 1 | 2 | 3 | 3 | 4 |

Lọc `<= 3`: `ROW_NUMBER` giữ 3 người, `RANK` giữ 3 người (100, 100, 90), `DENSE_RANK` giữ **5** người (100, 100, 90, 80, 80) — chỉ `DENSE_RANK` đúng với "top 3 mức lương phân biệt" (100, 90, 80).

### Bài 2

| e1 | Mức phân biệt cao hơn | Số đếm | Kết quả |
|---|---|:---:|:---:|
| An 3000 | 5000, 4000 | 2 | giữ |
| Bình 5000 | — | 0 | giữ |
| Chi 4000 | 5000 | 1 | giữ |
| Dũng 5000 | — | 0 | giữ |
| Em 2000 | 5000, 4000, 3000 | 3 | loại |
| Giang 4000 | 5000 | 1 | giữ |

→ 5 người: An, Bình, Chi, Dũng, Giang. Cell dưới kiểm chứng cả bài 1 và bài 2 bằng SQL.

In [38]:
# Kiểm chứng Bài 1
db_b1 = make_db([(i, f"P{i}", s, 1) for i, s in enumerate([100, 100, 90, 80, 80, 70], start=1)], [(1, "X")])
cols, rows = run(db_b1, """
    SELECT salary,
           ROW_NUMBER() OVER (ORDER BY salary DESC, id) AS rn,
           RANK()       OVER (ORDER BY salary DESC)     AS rk,
           DENSE_RANK() OVER (ORDER BY salary DESC)     AS dr
    FROM Employee ORDER BY salary DESC, id""")
assert [r[1] for r in rows] == [1, 2, 3, 4, 5, 6]
assert [r[2] for r in rows] == [1, 1, 3, 4, 4, 6]
assert [r[3] for r in rows] == [1, 1, 2, 3, 3, 4]
print("Bài 1: PASS, DENSE_RANK<=3 giữ", sum(r[3] <= 3 for r in rows), "người")

# Kiểm chứng Bài 2
emps_b2 = [(1, "An", 3000, 1), (2, "Bình", 5000, 1), (3, "Chi", 4000, 1),
           (4, "Dũng", 5000, 1), (5, "Em", 2000, 1), (6, "Giang", 4000, 1)]
db_b2 = make_db(emps_b2, [(1, "Ops")])
show(db_b2, """SELECT e1.name, e1.salary,
                      (SELECT COUNT(DISTINCT e2.salary) FROM Employee e2
                       WHERE e2.departmentId = e1.departmentId AND e2.salary > e1.salary) AS cnt
               FROM Employee e1 ORDER BY e1.id""")
check_query("Bài 2", db_b2, SQL_CORRELATED,
            [("Ops", n, s) for (_, n, s, _) in emps_b2 if n != "Em"])

Bài 1: PASS, DENSE_RANK<=3 giữ 5 người
+-------+--------+-----+
| name  | salary | cnt |
+-------+--------+-----+
| An    | 3000   | 2   |
| Bình  | 5000   | 0   |
| Chi   | 4000   | 1   |
| Dũng  | 5000   | 0   |
| Em    | 2000   | 3   |
| Giang | 4000   | 1   |
+-------+--------+-----+
Bài 2: PASS (5 hàng)


In [39]:
# Bài 3 — LeetCode 176
def answer_second_highest_sql() -> str:
    return """
    SELECT (SELECT DISTINCT salary
            FROM Employee
            ORDER BY salary DESC
            LIMIT 1 OFFSET 1) AS SecondHighestSalary
    """

for sal, exp in [([100, 200, 300], 200), ([100], None), ([100, 100], None), ([300, 300, 200], 200)]:
    assert run(make_salary_db(sal), answer_second_highest_sql()) == (["SecondHighestSalary"], [(exp,)])
print("Bài 3: PASS")


# Bài 4 — LeetCode 177 (hai phiên bản)
def answer_nth_highest_offset(db, n: int):
    return db.execute("SELECT (SELECT DISTINCT salary FROM Employee "
                      "ORDER BY salary DESC LIMIT 1 OFFSET ?)", (n - 1,)).fetchone()[0]


def answer_nth_highest_dense(db, n: int):
    return db.execute("""SELECT MAX(salary) FROM (
                             SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS r
                             FROM Employee) t
                         WHERE r = ?""", (n,)).fetchone()[0]


for fn in (answer_nth_highest_offset, answer_nth_highest_dense):
    db_177 = make_salary_db([100, 200, 300])
    assert fn(db_177, 1) == 300 and fn(db_177, 2) == 200 and fn(db_177, 4) is None
    assert fn(make_salary_db([100]), 2) is None
    assert fn(make_salary_db([200, 200, 100]), 2) == 100
    print("Bài 4:", fn.__name__, "PASS")

print("""Bản nộp LeetCode 177 (MySQL):
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  SET N = N - 1;
  RETURN (SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET N);
END""")

Bài 3: PASS
Bài 4: answer_nth_highest_offset PASS
Bài 4: answer_nth_highest_dense PASS
Bản nộp LeetCode 177 (MySQL):
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  SET N = N - 1;
  RETURN (SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET N);
END


In [40]:
# Bài 5 — LeetCode 184
def answer_dept_highest_window_sql() -> str:
    return SQL_WINDOW.replace("rnk <= 3", "rnk = 1")


def answer_dept_highest_no_window_sql() -> str:
    return """
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
    WHERE e.salary = (SELECT MAX(e2.salary) FROM Employee e2
                      WHERE e2.departmentId = e.departmentId)
    """

db_184 = make_db(EMPLOYEES_184, DEPARTMENTS_184)
check_query("Bài 5 - window", db_184, answer_dept_highest_window_sql(), EXPECTED_184)
check_query("Bài 5 - no window", db_184, answer_dept_highest_no_window_sql(), EXPECTED_184)
top1_edge = [r for r in EXPECTED_EDGE if r[2] == max(x[2] for x in EXPECTED_EDGE if x[0] == r[0])]
check_query("Bài 5 - window (edge)", db_edge, answer_dept_highest_window_sql(), top1_edge)
check_query("Bài 5 - no window (edge)", db_edge, answer_dept_highest_no_window_sql(), top1_edge)


# Bài 6 — thêm SalaryRank, không window function
def answer_rank_no_window_sql() -> str:
    return """
    WITH t AS (
        SELECT d.name AS Department, e1.name AS Employee, e1.salary AS Salary,
               1 + (SELECT COUNT(DISTINCT e2.salary)
                    FROM Employee e2
                    WHERE e2.departmentId = e1.departmentId
                      AND e2.salary > e1.salary) AS SalaryRank
        FROM Employee e1
        JOIN Department d ON e1.departmentId = d.id
    )
    SELECT Department, Employee, Salary, SalaryRank
    FROM t
    WHERE SalaryRank <= 3
    ORDER BY Department, SalaryRank, Employee
    """

expected_6 = [("HR", "Lan", 50000, 1),
              ("IT", "Max", 90000, 1), ("IT", "Joe", 85000, 2), ("IT", "Randy", 85000, 2), ("IT", "Will", 70000, 3),
              ("Ops", "Minh", 40000, 1), ("Ops", "Nam", 40000, 1), ("Ops", "Oanh", 40000, 1),
              ("RnD", "Phong", 90000, 1), ("RnD", "Quan", 90000, 1), ("RnD", "Son", 80000, 2),
              ("RnD", "Tam", 70000, 3), ("RnD", "Uyen", 70000, 3),
              ("Sales", "Henry", 80000, 1), ("Sales", "Sam", 60000, 2)]
cols, rows = run(db_edge, answer_rank_no_window_sql())
assert cols == ["Department", "Employee", "Salary", "SalaryRank"] and rows == expected_6
print("Bài 6: PASS (đúng cả thứ tự)")

Bài 5 - window: PASS (3 hàng)
Bài 5 - no window: PASS (3 hàng)
Bài 5 - window (edge): PASS (8 hàng)
Bài 5 - no window (edge): PASS (8 hàng)
Bài 6: PASS (đúng cả thứ tự)


## 18. Tổng kết: template ngắn và checklist trước khi submit

### Template "Top-N mỗi nhóm" nên thuộc lòng

```sql
WITH ranked AS (
    SELECT <các cột cần>,
           DENSE_RANK() OVER (PARTITION BY <cột nhóm>
                              ORDER BY <cột so sánh> DESC) AS rnk
    FROM <bảng chính>
    JOIN <bảng tra tên> ON <khóa ngoại> = <khóa chính>
)
SELECT <các cột cần> FROM ranked WHERE rnk <= N;
```

Bản không window function (MySQL cũ):

```sql
... WHERE (SELECT COUNT(DISTINCT x.<cột so sánh>)
           FROM <bảng> x
           WHERE x.<cột nhóm> = t.<cột nhóm>
             AND x.<cột so sánh> > t.<cột so sánh>) < N
```

Câu thần chú: **"Top N *mức* thì `DENSE_RANK`; hạng = 1 + số mức cao hơn; lọc hạng thì bọc CTE."**

### Checklist trước khi submit

- [ ] Đề nói "unique / distinct" → dùng `DENSE_RANK` hoặc `COUNT(DISTINCT ...)`, **không** `RANK`, `ROW_NUMBER`, `COUNT(...)` trần.
- [ ] Có `PARTITION BY` / điều kiện "cùng phòng" → xếp hạng **theo từng nhóm**, không phải toàn bảng; không dùng `LIMIT` để lấy top mỗi nhóm.
- [ ] Phép so sánh đúng chiều và đúng biên: `ORDER BY salary DESC`, `rnk <= 3` hoặc `count < 3`.
- [ ] Phòng ít hơn 3 mức lương, phòng một người, mọi người cùng lương vẫn ra đủ người (không so sánh với `NULL`).
- [ ] Self-join dùng `LEFT JOIN` để không mất người cao nhất.
- [ ] Không lọc window function trong `WHERE` của cùng câu → bọc CTE/bảng con.
- [ ] Ghi rõ `e.name` / `d.name` khi hai bảng trùng tên cột.
- [ ] Tên cột kết quả đúng chữ: `Department`, `Employee`, `Salary`.
- [ ] Thử trên ví dụ đề **và** dữ liệu tự chế có hòa lương (như bộ edge case mục 11).